In [2]:

"""
MetaRouteNet: Dual-Dataset Full Pipeline
==========================================
Datasets:
  1. Zenodo 5807719  — hERG Dataset (8879 SMILES + pIC50)
  2. CToxPred/PMC12756696 — issararab/CToxPred (cardiotoxicity labels)

Fingerprints (ONLY from specified links — no Morgan/MACCS/RDKit FP):
  1. Atom Pair          (RDKit AtomPairs.Pairs   — chemfp-compatible)
  2. Topological Torsion(RDKit AtomPairs.Torsions — chemfp-compatible)
  3. Avalon             (rdkit.Avalon.pyAvalonTools)
  4. SECFP              (RDKit MHFPEncoder — MolForge-compatible)
  5. Pretrained graph   (GIN pretrained on ZINC — MultiFG-compatible)
  6. Sequence embeddings(char-LSTM on SMILES   — CToxPred-compatible)

Split   : Train 70% | Val 15% | Test 15% (stratified, devset at pIC50=4.5)
Tuning  : Optuna (hyperparameter) | 2-Phase fine-tune | LR scheduler
Metrics : ROC-AUC, PR-AUC, Accuracy, F1, Precision, Recall,
          Brier Score, RMSE, MAE  — Target ≥ 0.87

Install:
  pip install torch torch-geometric rdkit-pypi numpy pandas
  pip install scikit-learn optuna requests tqdm matplotlib mhfp
"""

'\nMetaRouteNet: Dual-Dataset Full Pipeline\n==========================================\nDatasets:\n  1. Zenodo 5807719  — hERG Dataset (8879 SMILES + pIC50)\n  2. CToxPred/PMC12756696 — issararab/CToxPred (cardiotoxicity labels)\n\nFingerprints (ONLY from specified links — no Morgan/MACCS/RDKit FP):\n  1. Atom Pair          (RDKit AtomPairs.Pairs   — chemfp-compatible)\n  2. Topological Torsion(RDKit AtomPairs.Torsions — chemfp-compatible)\n  3. Avalon             (rdkit.Avalon.pyAvalonTools)\n  4. SECFP              (RDKit MHFPEncoder — MolForge-compatible)\n  5. Pretrained graph   (GIN pretrained on ZINC — MultiFG-compatible)\n  6. Sequence embeddings(char-LSTM on SMILES   — CToxPred-compatible)\n\nSplit   : Train 70% | Val 15% | Test 15% (stratified, devset at pIC50=4.5)\nTuning  : Optuna (hyperparameter) | 2-Phase fine-tune | LR scheduler\nMetrics : ROC-AUC, PR-AUC, Accuracy, F1, Precision, Recall,\n          Brier Score, RMSE, MAE  — Target ≥ 0.87\n\nInstall:\n  pip install torch

In [3]:
import sys
print(sys.executable)  # Should point to your pythoncore-3.14-64 path

import torch
import torch_geometric
import rdkit
print("All packages loaded successfully!")

C:\Users\mruna\AppData\Local\Python\pythoncore-3.14-64\python.exe
All packages loaded successfully!


In [4]:
# ─────────────────────────────────────────────────────────────────────
# STEP 1: Imports & Config
# ─────────────────────────────────────────────────────────────────────
import os, io, warnings, requests, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch_geometric.nn import GINConv, GATConv, global_mean_pool
from torch_geometric.data import Data, Batch

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
from rdkit.Chem.AtomPairs import Pairs, Torsions
from rdkit.Avalon import pyAvalonTools as AvalonTools
from rdkit.Chem.rdMHFPFingerprint import MHFPEncoder as RDKitMHFP

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    f1_score, precision_score, recall_score,
    brier_score_loss, mean_squared_error, mean_absolute_error
)
import optuna
from optuna.pruners import MedianPruner
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

# ── Dimension registry for all 6 fingerprint types ──────────────────
FP_DIMS = {
    "atom_pair":            2048,
    "topological_torsion":  2048,
    "avalon":               1024,
    "secfp":                2048,
    "graph_embed":           300,   # GIN pretrained dim
    "seq_embed":             128,   # char-LSTM dim
}
# Combined FP dim (all 6 concatenated)
COMBINED_DIM = sum(FP_DIMS.values())   # 7552
RDKIT_DIM    = 200


Device: cpu


In [5]:
# STEP 1 · DATA INGESTION

import pandas as pd

df = pd.read_csv(r"C:\Users\mruna\OneDrive - Northumbria University - Production Azure AD\W24034825Dessertation\5807719\hERG_Dataset.csv")
df.head()

,SourceID,SMILES,PIC50,USED_AS
0,CHEMBL428777,C[C@@H](Oc1cccc2ncnc(Nc3ccc(OCc4ccccn4)c(Cl)c3...,4.85,Dev-set
1,CHEMBL258313,C[C@@H](Oc1cccc2ncnc(Nc3ccc(OCc4ccccn4)c(Cl)c3...,4.92,Dev-set
2,CHEMBL403774,C[C@@H](Oc1cccc2ncnc(Nc3ccc(OCc4ccccn4)c(Cl)c3...,5.00,Dev-set
3,CHEMBL402011,C[C@@H](Oc1cccc2ncnc(Nc3ccc4c(cnn4Cc5ccccn5)c3...,4.60,Dev-set
4,CHEMBL379816,C[C@@H](O[C@H]1CCNC(=O)C[C@@H]1c2ccc(F)cc2)c3c...,5.00,Dev-set


In [6]:
# STEP 3 · LABEL ENCODING + SAVE

df["hERG_Class"] = df["PIC50"].apply(lambda x: "Blocker" if x >= 4.5 else "Non-Blocker")

# Save modified dataset
save_path = r"C:\Users\mruna\OneDrive - Northumbria University - Production Azure AD\W24034825Dessertation\5807719\hERG_Dataset_Labeled.csv"
df.to_csv(save_path, index=False)

# Show updated dataframe with new column
print(df.head())
print(f"\n Modified dataset saved at:\n{save_path}")

# Pie chart
counts = df["hERG_Class"].value_counts()
pie_labels = ["Blocker\n(PIC50 ≥ 4.5)", "Non-Blocker\n(PIC50 < 4.5)"]

plt.figure(figsize=(6, 6))
plt.pie(counts, labels=pie_labels, autopct="%1.1f%%", startangle=140)
plt.title("hERG Blocker Distribution")
plt.tight_layout()
plt.show()

       SourceID                                             SMILES  PIC50  \
0  CHEMBL428777  C[C@@H](Oc1cccc2ncnc(Nc3ccc(OCc4ccccn4)c(Cl)c3...   4.85   
1  CHEMBL258313  C[C@@H](Oc1cccc2ncnc(Nc3ccc(OCc4ccccn4)c(Cl)c3...   4.92   
2  CHEMBL403774  C[C@@H](Oc1cccc2ncnc(Nc3ccc(OCc4ccccn4)c(Cl)c3...   5.00   
3  CHEMBL402011  C[C@@H](Oc1cccc2ncnc(Nc3ccc4c(cnn4Cc5ccccn5)c3...   4.60   
4  CHEMBL379816  C[C@@H](O[C@H]1CCNC(=O)C[C@@H]1c2ccc(F)cc2)c3c...   5.00   

   USED_AS hERG_Class  
0  Dev-set    Blocker  
1  Dev-set    Blocker  
2  Dev-set    Blocker  
3  Dev-set    Blocker  
4  Dev-set    Blocker  

 Modified dataset saved at:
C:\Users\mruna\OneDrive - Northumbria University - Production Azure AD\W24034825Dessertation\5807719\hERG_Dataset_Labeled.csv


In [7]:
# STEP 4 · DATA QUALITY CHECK + FEATURE ENGINEERING

print("=" * 60)
print("        DATA QUALITY REPORT")
print("=" * 60)

# ── 1. BASIC INFO ─────────────────────────────────────────────
print(f"\n Dataset Shape     : {df.shape}")
print(f" Columns           : {df.columns.tolist()}")
print(f"\n Data Types:\n{df.dtypes}")

# ── 2. MISSING VALUES ─────────────────────────────────────────
print("\n" + "=" * 60)
print("  MISSING VALUES")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print(missing_df[missing_df["Missing Count"] > 0])
if missing_df["Missing Count"].sum() == 0:
    print(" No missing values found!")

# ── 3. DUPLICATE ROWS ─────────────────────────────────────────
print("\n" + "=" * 60)
print("  DUPLICATE ROWS")
print("=" * 60)
total_dupes = df.duplicated().sum()
smiles_dupes = df.duplicated(subset=["SMILES"]).sum()
print(f" Total duplicate rows     : {total_dupes}")
print(f" Duplicate SMILES entries : {smiles_dupes}")

# Drop duplicates based on SMILES
df = df.drop_duplicates(subset=["SMILES"]).reset_index(drop=True)
print(f" After dropping duplicates: {df.shape[0]} rows remaining")

# ── 4. CLASS IMBALANCE ────────────────────────────────────────
print("\n" + "=" * 60)
print("  CLASS DISTRIBUTION (hERG_Class)")
print("=" * 60)
class_counts = df["hERG_Class"].value_counts()
class_pct    = df["hERG_Class"].value_counts(normalize=True) * 100
class_df     = pd.DataFrame({"Count": class_counts, "Percentage %": class_pct})
print(class_df)

ratio = class_counts.max() / class_counts.min()
if ratio > 1.5:
    print(f"\n  Imbalance Ratio: {ratio:.2f} → Consider SMOTE or class weighting!")
else:
    print(f"\n Classes are balanced (ratio: {ratio:.2f})")

# Visualise class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
axes[0].pie(class_counts, labels=class_counts.index,
            autopct="%1.1f%%", startangle=140,
            colors=["#2196F3", "#FF5722"])
axes[0].set_title("Class Distribution (Pie)")

# Bar chart
axes[1].bar(class_counts.index, class_counts.values,
            color=["#2196F3", "#FF5722"], edgecolor="black")
axes[1].set_title("Class Distribution (Bar)")
axes[1].set_ylabel("Count")
for i, v in enumerate(class_counts.values):
    axes[1].text(i, v + 5, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# ── 5. PIC50 DISTRIBUTION ─────────────────────────────────────
print("\n" + "=" * 60)
print("  PIC50 STATISTICAL SUMMARY")
print("=" * 60)
print(df["PIC50"].describe())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df["PIC50"], bins=40, color="#4CAF50", edgecolor="black")
plt.axvline(4.5, color="red", linestyle="--", label="Threshold (4.5)")
plt.title("PIC50 Distribution")
plt.xlabel("PIC50")
plt.ylabel("Frequency")
plt.legend()

plt.subplot(1, 2, 2)
plt.boxplot(df["PIC50"], vert=True, patch_artist=True,
            boxprops=dict(facecolor="#4CAF50", color="black"))
plt.title("PIC50 Boxplot (Outlier Check)")
plt.ylabel("PIC50")
plt.tight_layout()
plt.show()

# ── 6. SMILES VALIDITY CHECK ──────────────────────────────────
print("\n" + "=" * 60)
print("  SMILES VALIDITY CHECK")
print("=" * 60)
df["valid_mol"] = df["SMILES"].apply(lambda s: Chem.MolFromSmiles(s) is not None)
invalid = df[df["valid_mol"] == False]
print(f" Valid SMILES   : {df['valid_mol'].sum()}")
print(f" Invalid SMILES : {len(invalid)}")

# Drop invalid SMILES
df = df[df["valid_mol"] == True].reset_index(drop=True)
df.drop(columns=["valid_mol"], inplace=True)
print(f" Clean dataset shape: {df.shape}")

# ── 7. FEATURE ENGINEERING ────────────────────────────────────
print("\n" + "=" * 60)
print("  FEATURE ENGINEERING")
print("=" * 60)
from rdkit.Chem import Descriptors, rdMolDescriptors

def extract_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "MolWt"        : Descriptors.MolWt(mol),
        "LogP"         : Descriptors.MolLogP(mol),
        "NumHDonors"   : rdMolDescriptors.CalcNumHBD(mol),
        "NumHAcceptors": rdMolDescriptors.CalcNumHBA(mol),
        "NumRings"      : rdMolDescriptors.CalcNumRings(mol),
        "NumAromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "TPSA"         : Descriptors.TPSA(mol),
        "NumRotBonds"  : rdMolDescriptors.CalcNumRotatableBonds(mol),
        "NumHeavyAtoms": mol.GetNumHeavyAtoms(),
        "FractionCSP3" : rdMolDescriptors.CalcFractionCSP3(mol),
    }

feat_df = df["SMILES"].apply(extract_features).apply(pd.Series)
df      = pd.concat([df, feat_df], axis=1)

print(f" Added {feat_df.shape[1]} engineered features")
print(df.head())

# ── FINAL SUMMARY ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("  FINAL CLEAN DATASET SUMMARY")
print("=" * 60)
print(f" Shape            : {df.shape}")
print(f" Unique SMILES    : {df['SMILES'].nunique()}")
print(f"  Class Balance    : {df['hERG_Class'].value_counts().to_dict()}")
print(" Data Quality Check Complete!")

        DATA QUALITY REPORT

 Dataset Shape     : (8879, 5)
 Columns           : ['SourceID', 'SMILES', 'PIC50', 'USED_AS', 'hERG_Class']

 Data Types:
SourceID          str
SMILES            str
PIC50         float64
USED_AS           str
hERG_Class        str
dtype: object

  MISSING VALUES
Empty DataFrame
Columns: [Missing Count, Missing %]
Index: []
 No missing values found!

  DUPLICATE ROWS
 Total duplicate rows     : 0
 Duplicate SMILES entries : 1
 After dropping duplicates: 8878 rows remaining

  CLASS DISTRIBUTION (hERG_Class)
             Count  Percentage %
hERG_Class                      
Blocker       7385     83.183149
Non-Blocker   1493     16.816851

  Imbalance Ratio: 4.95 → Consider SMOTE or class weighting!

  PIC50 STATISTICAL SUMMARY
count    8878.000000
mean        5.283920
std         0.979232
min         0.000000
25%         4.600000
50%         5.085000
75%         5.750000
max         9.850000
Name: PIC50, dtype: float64

  SMILES VALIDITY CHECK
 Valid SMILES

In [8]:
# STEP 5 · COMPLETE MOLECULAR PREPROCESSING PIPELINE

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, MolStandardize
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import inchi
import numpy as np
import pandas as pd

print("=" * 60)
print("   MOLECULAR PREPROCESSING PIPELINE")
print("=" * 60)

initial_count = len(df)

# ── 1. MIXTURE DETECTION (multi-fragment SMILES) ──────────────
print("\n Step 1: Mixture Detection")
df["is_mixture"] = df["SMILES"].apply(lambda s: "." in str(s))
mixtures = df["is_mixture"].sum()
print(f"  Mixtures/salts detected : {mixtures}")
df = df[df["is_mixture"] == False].reset_index(drop=True)
df.drop(columns=["is_mixture"], inplace=True)
print(f" Rows after removal       : {len(df)}")

# ── 2. INORGANIC / METAL REMOVAL ──────────────────────────────
print("\n Step 2: Inorganic & Metal Compound Removal")
organic_atoms = {"C", "N", "O", "S", "F", "P", "Cl", "Br", "I", "H"}

def is_organic(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    atoms = {atom.GetSymbol() for atom in mol.GetAtoms()}
    return "C" in atoms and atoms.issubset(organic_atoms)

df["is_organic"] = df["SMILES"].apply(is_organic)
inorganic_count = (~df["is_organic"]).sum()
print(f"  Inorganic/metal compounds: {inorganic_count}")
df = df[df["is_organic"] == True].reset_index(drop=True)
df.drop(columns=["is_organic"], inplace=True)
print(f" Rows after removal        : {len(df)}")

# ── 3. SALT STRIPPING & NEUTRALIZATION ────────────────────────
print("\n Step 3: Salt Stripping & Neutralization")
remover     = rdMolStandardize.FragmentParent
normalizer  = rdMolStandardize.Normalizer()
uncharger   = rdMolStandardize.Uncharger()

def strip_and_neutralize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = rdMolStandardize.FragmentParent(mol)   # keep largest fragment
    mol = normalizer.normalize(mol)              # normalize bonds
    mol = uncharger.uncharge(mol)                # neutralize charges
    return Chem.MolToSmiles(mol)

df["SMILES"] = df["SMILES"].apply(strip_and_neutralize)
df = df[df["SMILES"].notna()].reset_index(drop=True)
print(f" Rows after stripping      : {len(df)}")

# ── 4. SMILES CANONICALIZATION ────────────────────────────────
print("\n Step 4: SMILES Canonicalization")

def canonicalize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

df["SMILES"] = df["SMILES"].apply(canonicalize)
df = df[df["SMILES"].notna()].reset_index(drop=True)
print(f" Canonical SMILES applied  : {len(df)}")

# ── 5. TAUTOMER STANDARDIZATION ───────────────────────────────
print("\n Step 5: Tautomer Standardization")
tautomer_enumerator = rdMolStandardize.TautomerEnumerator()

def standardize_tautomer(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    canon = tautomer_enumerator.Canonicalize(mol)
    return Chem.MolToSmiles(canon)

df["SMILES"] = df["SMILES"].apply(standardize_tautomer)
df = df[df["SMILES"].notna()].reset_index(drop=True)
print(f" Tautomers standardized    : {len(df)}")

# ── 6. INCHIKEY DEDUPLICATION (more robust than SMILES) ───────
print("\n Step 6: InChIKey-based Deduplication")

def get_inchikey(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return inchi.MolToInchiKey(mol)

df["InChIKey"] = df["SMILES"].apply(get_inchikey)
dupes = df.duplicated(subset=["InChIKey"]).sum()
print(f"  Duplicate InChIKeys      : {dupes}")
df = df.drop_duplicates(subset=["InChIKey"]).reset_index(drop=True)
print(f" Rows after dedup          : {len(df)}")

# ── 7. OUTLIER REMOVAL IN PIC50 ───────────────────────────────
print("\n Step 7: PIC50 Outlier Removal (IQR Method)")
Q1  = df["PIC50"].quantile(0.25)
Q3  = df["PIC50"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = df[(df["PIC50"] < lower) | (df["PIC50"] > upper)]
print(f"  Outliers detected        : {len(outliers)}")
print(f"   PIC50 valid range        : [{lower:.2f} → {upper:.2f}]")
df = df[(df["PIC50"] >= lower) & (df["PIC50"] <= upper)].reset_index(drop=True)
print(f" Rows after removal        : {len(df)}")

# ── 8. DRUG-LIKENESS FILTER (Lipinski + Veber) ────────────────
print("\n Step 8: Drug-likeness Filter (Lipinski + Veber)")

def drug_likeness(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    mw    = Descriptors.MolWt(mol)
    logp  = Descriptors.MolLogP(mol)
    hbd   = rdMolDescriptors.CalcNumHBD(mol)
    hba   = rdMolDescriptors.CalcNumHBA(mol)
    tpsa  = Descriptors.TPSA(mol)
    rotb  = rdMolDescriptors.CalcNumRotatableBonds(mol)
    # Lipinski Ro5
    lipinski = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)
    # Veber rules
    veber    = (tpsa <= 140) and (rotb <= 10)
    return lipinski and veber

df["drug_like"] = df["SMILES"].apply(drug_likeness)
non_drug = (~df["drug_like"]).sum()
print(f"  Non drug-like compounds  : {non_drug}")
df = df[df["drug_like"] == True].reset_index(drop=True)
df.drop(columns=["drug_like"], inplace=True)
print(f" Rows after filtering      : {len(df)}")

# ── 9. STEREOCHEMISTRY FLAG ───────────────────────────────────
print("\n Step 9: Stereochemistry Annotation")

def has_stereo(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    return any(atom.GetChiralTag() != Chem.CHI_UNSPECIFIED
               for atom in mol.GetAtoms())

df["has_stereo"] = df["SMILES"].apply(has_stereo)
print(f" Molecules with stereocenters : {df['has_stereo'].sum()}")
print(f"   (kept as metadata — not removed)")

# ── 10. SCAFFOLD ANALYSIS (Bemis-Murcko) ──────────────────────
print("\n Step 10: Bemis-Murcko Scaffold Diversity")
from rdkit.Chem.Scaffolds import MurckoScaffold

def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)

df["Scaffold"] = df["SMILES"].apply(get_scaffold)
unique_scaffolds = df["Scaffold"].nunique()
print(f"  Unique Bemis-Murcko scaffolds: {unique_scaffolds}")
print(f"   Scaffold diversity ratio     : {unique_scaffolds/len(df):.2%}")

# ── FINAL PREPROCESSING SUMMARY ───────────────────────────────
print("\n" + "=" * 60)
print("   PREPROCESSING COMPLETE — SUMMARY")
print("=" * 60)
print(f" Initial molecules         : {initial_count}")
print(f" Final clean molecules     : {len(df)}")
print(f"  Total removed             : {initial_count - len(df)}")
print(f"  Class balance             : {df['hERG_Class'].value_counts().to_dict()}")
print(f" Stereocenters present      : {df['has_stereo'].sum()}")
print(f"  Scaffold diversity         : {unique_scaffolds} unique scaffolds")
print(" Ready for fingerprint generation!")

   MOLECULAR PREPROCESSING PIPELINE

 Step 1: Mixture Detection
  Mixtures/salts detected : 0
 Rows after removal       : 8878

 Step 2: Inorganic & Metal Compound Removal
  Inorganic/metal compounds: 6
 Rows after removal        : 8872

 Step 3: Salt Stripping & Neutralization


[15:40:15] Initializing Normalizer
[15:40:15] Initializing MetalDisconnector
[15:40:15] Running MetalDisconnector
[15:40:15] Initializing Normalizer
[15:40:15] Running Normalizer
[15:40:15] Running LargestFragmentChooser
[15:40:15] Running Normalizer
[15:40:15] Running Uncharger
[15:40:15] Initializing MetalDisconnector
[15:40:15] Running MetalDisconnector
[15:40:15] Initializing Normalizer
[15:40:15] Running Normalizer
[15:40:15] Running LargestFragmentChooser
[15:40:15] Running Normalizer
[15:40:15] Running Uncharger
[15:40:15] Initializing MetalDisconnector
[15:40:15] Running MetalDisconnector
[15:40:15] Initializing Normalizer
[15:40:15] Running Normalizer
[15:40:15] Running LargestFragmentChooser
[15:40:15] Running Normalizer
[15:40:15] Running Uncharger
[15:40:15] Initializing MetalDisconnector
[15:40:15] Running MetalDisconnector
[15:40:15] Initializing Normalizer
[15:40:15] Running Normalizer
[15:40:15] Running LargestFragmentChooser
[15:40:15] Running Normalizer
[15:40:15] Run

 Rows after stripping      : 8872

 Step 4: SMILES Canonicalization
 Canonical SMILES applied  : 8872

 Step 5: Tautomer Standardization


[15:40:40] Can't kekulize mol.  Unkekulized atoms: 3 7
[15:40:43] Tautomer enumeration stopped at 406 tautomers: max transforms reached
[15:40:43] Can't kekulize mol.  Unkekulized atoms: 2 32
[15:40:45] Tautomer enumeration stopped at 200 tautomers: max transforms reached
[15:40:45] Can't kekulize mol.  Unkekulized atoms: 2 34
[15:40:45] Can't kekulize mol.  Unkekulized atoms: 2 34
[15:41:06] Can't kekulize mol.  Unkekulized atoms: 2 31
[15:41:06] Can't kekulize mol.  Unkekulized atoms: 2 31
[15:41:07] Tautomer enumeration stopped at 343 tautomers: max transforms reached
[15:41:09] Can't kekulize mol.  Unkekulized atoms: 3 5 6 10
[15:41:10] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 11
[15:41:10] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 11
[15:41:10] Can't kekulize mol.  Unkekulized atoms: 3 7
[15:41:10] Can't kekulize mol.  Unkekulized atoms: 2 23
[15:41:11] Can't kekulize mol.  Unkekulized atoms: 2 23
[15:41:11] Can't kekulize mol.  Unkekulized atoms: 2 23
[15:41:11] 

 Tautomers standardized    : 8872

 Step 6: InChIKey-based Deduplication
  Duplicate InChIKeys      : 167
 Rows after dedup          : 8705

 Step 7: PIC50 Outlier Removal (IQR Method)
  Outliers detected        : 385
   PIC50 valid range        : [2.89 → 7.45]
 Rows after removal        : 8320

 Step 8: Drug-likeness Filter (Lipinski + Veber)
  Non drug-like compounds  : 2589
 Rows after filtering      : 5731

 Step 9: Stereochemistry Annotation
 Molecules with stereocenters : 1956
   (kept as metadata — not removed)

 Step 10: Bemis-Murcko Scaffold Diversity
  Unique Bemis-Murcko scaffolds: 3033
   Scaffold diversity ratio     : 52.92%

   PREPROCESSING COMPLETE — SUMMARY
 Initial molecules         : 8878
 Final clean molecules     : 5731
  Total removed             : 3147
  Class balance             : {'Blocker': 4663, 'Non-Blocker': 1068}
 Stereocenters present      : 1956
  Scaffold diversity         : 3033 unique scaffolds
 Ready for fingerprint generation!


In [9]:
# STEP 6 · LABEL ENCODING ON CLEAN DATA

df["hERG_Class"] = df["PIC50"].apply(lambda x: "Blocker" if x >= 4.5 else "Non-Blocker")
df["Label"]      = df["hERG_Class"].map({"Blocker": 1, "Non-Blocker": 0})

print("=" * 60)
print("  LABEL ENCODING COMPLETE")
print("=" * 60)
print(df[["SMILES", "PIC50", "hERG_Class", "Label"]].head(10))
print(f"\nTotal labeled molecules : {len(df)}")

  LABEL ENCODING COMPLETE
                                              SMILES  PIC50 hERG_Class  Label
0  CC(Oc1cccc2ncnc(Nc3ccc(OCc4ccccn4)c(Cl)c3)c12)...   4.85    Blocker      1
1  CC(Oc1cccc2ncnc(Nc3ccc4c(cnn4Cc4ccccn4)c3)c12)...   4.60    Blocker      1
2         Cc1[nH]ncc1-c1ccc2cc(CCN3CCC[C@H]3C)ccc2n1   4.75    Blocker      1
3   Cc1nc(C)c(-c2ccc3cc(CCN4CCC[C@H]4C)ccc3n2)cc1C#N   5.70    Blocker      1
4  Cc1nc(C)c(-c2ccc3cc(CCN4CCC[C@H]4C)ccc3n2)cc1C...   5.00    Blocker      1
5  Cc1ccc(Oc2ccc(Nc3ncnc4cccc(OC(C)C(=O)N(C)CCO)c...   4.92    Blocker      1
6  C[C@@H]1CCCN1C(=O)c1ccc(-c2ccc3c(c2)CCN(CCN2CC...   4.98    Blocker      1
7  C[C@@H]1CCCN1CCc1ccc(-c2ccc(S(=O)(=O)CC3CCOCC3...   5.51    Blocker      1
8  C[C@@H]1[C@@H](c2ccc(OCCCN3CCCC3)cc2)Oc2ccccc2...   5.89    Blocker      1
9  C[C@@H](Oc1nc(-c2cnn(C(F)F)c2)cc2ncsc12)[C@H]1...   4.52    Blocker      1

Total labeled molecules : 5731


In [10]:
# STEP 7 · CLASS IMBALANCE ANALYSIS (RESPECTING PRE-DEFINED SPLITS)

print("=" * 60)
print("  CLASS IMBALANCE ANALYSIS")
print("=" * 60)

# Full dataset distribution
full_counts = df["hERG_Class"].value_counts()
full_pct    = df["hERG_Class"].value_counts(normalize=True) * 100
imbalance_ratio = full_counts.max() / full_counts.min()

print("\nFull Dataset:")
print(pd.DataFrame({"Count": full_counts, "Percentage": full_pct.round(2)}))
print(f"\nImbalance Ratio         : {imbalance_ratio:.2f}:1")

if imbalance_ratio > 3:
    print("Severe imbalance detected -- class weighting or oversampling required")
elif imbalance_ratio > 1.5:
    print("Moderate imbalance detected -- class weighting recommended")
else:
    print("Classes are balanced")

# Split-wise distribution using USED_AS column
print("\n" + "=" * 60)
print("  SPLIT-WISE CLASS DISTRIBUTION (USED_AS)")
print("=" * 60)

for split in df["USED_AS"].unique():
    subset = df[df["USED_AS"] == split]
    counts = subset["hERG_Class"].value_counts()
    pct    = subset["hERG_Class"].value_counts(normalize=True) * 100
    print(f"\n{split} ({len(subset)} molecules):")
    print(pd.DataFrame({"Count": counts, "Percentage": pct.round(2)}))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Full dataset pie
axes[0].pie(full_counts, labels=full_counts.index,
            autopct="%1.1f%%", startangle=140,
            colors=["#2196F3", "#FF5722"])
axes[0].set_title("Full Dataset Distribution")

# Split-wise bar chart
splits     = df["USED_AS"].unique()
blockers   = [df[(df["USED_AS"] == s) & (df["Label"] == 1)].shape[0] for s in splits]
nonblockers= [df[(df["USED_AS"] == s) & (df["Label"] == 0)].shape[0] for s in splits]

x = range(len(splits))
axes[1].bar(x, blockers,    label="Blocker",     color="#2196F3", edgecolor="black")
axes[1].bar(x, nonblockers, label="Non-Blocker", color="#FF5722",
            bottom=blockers, edgecolor="black")
axes[1].set_xticks(x)
axes[1].set_xticklabels(splits)
axes[1].set_title("Class Distribution by Split")
axes[1].set_ylabel("Count")
axes[1].legend()

# PIC50 distribution by class
for label, grp in df.groupby("hERG_Class"):
    axes[2].hist(grp["PIC50"], bins=40, alpha=0.6, label=label, edgecolor="black")
axes[2].axvline(4.5, color="red", linestyle="--", linewidth=1.5, label="Threshold (4.5)")
axes[2].set_title("PIC50 Distribution by Class")
axes[2].set_xlabel("PIC50")
axes[2].set_ylabel("Frequency")
axes[2].legend()

plt.tight_layout()
plt.show()

# Class weight computation for imbalanced training
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight = "balanced",
    classes      = np.array([0, 1]),
    y            = df["Label"].values
)

print("\n" + "=" * 60)
print("  COMPUTED CLASS WEIGHTS FOR MODEL TRAINING")
print("=" * 60)
print(f"Non-Blocker (0) weight  : {class_weights[0]:.4f}")
print(f"Blocker     (1) weight  : {class_weights[1]:.4f}")
print("\nNote: Use these weights directly in your loss function")
print("      during GNN, CNN, and FCNN training to handle imbalance.")

  CLASS IMBALANCE ANALYSIS

Full Dataset:
             Count  Percentage
hERG_Class                    
Blocker       4663       81.36
Non-Blocker   1068       18.64

Imbalance Ratio         : 4.37:1
Severe imbalance detected -- class weighting or oversampling required

  SPLIT-WISE CLASS DISTRIBUTION (USED_AS)

Dev-set (5484 molecules):
             Count  Percentage
hERG_Class                    
Blocker       4490       81.87
Non-Blocker    994       18.13

Test-set (247 molecules):
             Count  Percentage
hERG_Class                    
Blocker        173       70.04
Non-Blocker     74       29.96

  COMPUTED CLASS WEIGHTS FOR MODEL TRAINING
Non-Blocker (0) weight  : 2.6831
Blocker     (1) weight  : 0.6145

Note: Use these weights directly in your loss function
      during GNN, CNN, and FCNN training to handle imbalance.


In [11]:
# STEP 8 · PHYSICOCHEMICAL FEATURE ENGINEERING

from rdkit.Chem import Descriptors, rdMolDescriptors, GraphDescriptors
from rdkit.Chem import Fragments

print("=" * 60)
print("  PHYSICOCHEMICAL FEATURE ENGINEERING")
print("=" * 60)

def compute_physicochemical(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    features = {
        # Lipophilicity and solubility
        "MolLogP"              : Descriptors.MolLogP(mol),
        "MolMR"                : Descriptors.MolMR(mol),

        # Molecular size
        "MolWt"                : Descriptors.MolWt(mol),
        "HeavyAtomMolWt"       : Descriptors.HeavyAtomMolWt(mol),
        "NumHeavyAtoms"        : mol.GetNumHeavyAtoms(),

        # Hydrogen bonding
        "NumHDonors"           : rdMolDescriptors.CalcNumHBD(mol),
        "NumHAcceptors"        : rdMolDescriptors.CalcNumHBA(mol),

        # Topology
        "NumRotatableBonds"    : rdMolDescriptors.CalcNumRotatableBonds(mol),
        "NumRings"             : rdMolDescriptors.CalcNumRings(mol),
        "NumAromaticRings"     : rdMolDescriptors.CalcNumAromaticRings(mol),
        "NumSaturatedRings"    : rdMolDescriptors.CalcNumSaturatedRings(mol),
        "NumAliphaticRings"    : rdMolDescriptors.CalcNumAliphaticRings(mol),

        # Surface area
        "TPSA"                 : Descriptors.TPSA(mol),
        "LabuteASA"            : Descriptors.LabuteASA(mol),

        # Electronic properties
        "MaxPartialCharge"     : Descriptors.MaxPartialCharge(mol),
        "MinPartialCharge"     : Descriptors.MinPartialCharge(mol),
        "MaxAbsPartialCharge"  : Descriptors.MaxAbsPartialCharge(mol),

        # Complexity
        "BertzCT"              : GraphDescriptors.BertzCT(mol),
        "FractionCSP3"         : rdMolDescriptors.CalcFractionCSP3(mol),

        # Atom counts
        "NumNitrogens"         : sum(1 for a in mol.GetAtoms() if a.GetSymbol() == "N"),
        "NumOxygens"           : sum(1 for a in mol.GetAtoms() if a.GetSymbol() == "O"),
        "NumSulfurs"           : sum(1 for a in mol.GetAtoms() if a.GetSymbol() == "S"),
        "NumHalogens"          : sum(1 for a in mol.GetAtoms()
                                     if a.GetSymbol() in {"F", "Cl", "Br", "I"}),

        # Bond counts
        "NumAromaticBonds"     : sum(1 for b in mol.GetBonds()
                                     if b.GetIsAromatic()),
        "NumSingleBonds"       : sum(1 for b in mol.GetBonds()
                                     if b.GetBondTypeAsDouble() == 1.0),
        "NumDoubleBonds"       : sum(1 for b in mol.GetBonds()
                                     if b.GetBondTypeAsDouble() == 2.0),

        # Stereocenters
        "NumStereocenters"     : len(Chem.FindMolChiralCenters(mol, includeUnassigned=True)),

        # hERG-relevant fragments
        "NumBasicNitrogens"    : Fragments.fr_NH0(mol) +
                                 Fragments.fr_NH1(mol) +
                                 Fragments.fr_NH2(mol) +
                                 Fragments.fr_Ar_N(mol),
        "NumAromatics"         : Fragments.fr_benzene(mol),

        # Tertiary amines via SMARTS -- non-amide tertiary nitrogens
        "NumTertAmines"        : len(mol.GetSubstructMatches(
                                     Chem.MolFromSmarts("[N;X3;H0;!$(NC=O)]"))),
    }
    return features

print("Computing physicochemical features...")
feat_df = df["SMILES"].apply(compute_physicochemical).apply(pd.Series)

# Check for failed rows
failed = feat_df.isnull().all(axis=1).sum()
print(f"Failed computations     : {failed}")

# Merge into main dataframe
df = pd.concat([df.reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)

# Remove any duplicate columns introduced during concat
df = df.loc[:, ~df.columns.duplicated()].reset_index(drop=True)

print(f"Features added          : {feat_df.shape[1]}")
print(f"Updated dataset shape   : {df.shape}")

# Feature summary statistics by class
print("\n" + "=" * 60)
print("  FEATURE STATISTICS BY CLASS")
print("=" * 60)
feature_cols = feat_df.columns.tolist()
summary      = df.groupby("hERG_Class")[feature_cols].mean().T
summary.columns  = ["Blocker_Mean", "NonBlocker_Mean"]
summary["Difference"] = abs(summary["Blocker_Mean"] - summary["NonBlocker_Mean"])
summary = summary.sort_values("Difference", ascending=False)
print(summary.round(4))

# Correlation with Label
print("\n" + "=" * 60)
print("  TOP FEATURES CORRELATED WITH hERG ACTIVITY")
print("=" * 60)
correlations = df[feature_cols + ["Label"]].corr()["Label"].drop("Label")
correlations = correlations.abs().sort_values(ascending=False)
print(correlations.head(15).round(4))

# Visualize top 10 features by class
top_features = correlations.head(10).index.tolist()
fig, axes    = plt.subplots(2, 5, figsize=(20, 8))
axes         = axes.flatten()

for i, feat in enumerate(top_features):
    for label, grp in df.groupby("hERG_Class"):
        axes[i].hist(grp[feat].dropna(), bins=30, alpha=0.6,
                     label=label, edgecolor="black")
    axes[i].set_title(feat, fontsize=9)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")
    axes[i].legend(fontsize=7)

plt.suptitle("Top 10 Physicochemical Features by hERG Class", fontsize=12)
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("  STEP 9 COMPLETE -- READY FOR FINGERPRINT GENERATION")
print("=" * 60)
print(f"Total dataset shape     : {df.shape}")
print(f"Physicochemical features: {len(feature_cols)}")
print(f"Dev-set size            : {df[df['USED_AS'] == 'Dev-set'].shape[0]}")
print(f"Test-set size           : {df[df['USED_AS'] == 'Test-set'].shape[0]}")

  PHYSICOCHEMICAL FEATURE ENGINEERING
Computing physicochemical features...
Failed computations     : 0
Features added          : 30
Updated dataset shape   : (5731, 41)

  FEATURE STATISTICS BY CLASS
                     Blocker_Mean  NonBlocker_Mean  Difference
BertzCT                  906.1993         881.4387     24.7606
TPSA                      65.3785          77.8248     12.4463
HeavyAtomMolWt           365.3133         361.7606      3.5527
MolWt                    388.8779         385.4344      3.4435
MolMR                    106.5139         103.9645      2.5494
LabuteASA                164.0696         161.6292      2.4404
NumAromaticBonds          14.2072          12.6910      1.5162
MolLogP                    3.5307           2.7884      0.7423
NumSingleBonds            15.1621          15.8830      0.7208
NumOxygens                 1.9455           2.4644      0.5189
NumHAcceptors              4.8527           5.2781      0.4254
NumDoubleBonds             1.1231          

In [12]:
# STEP 9 · FINGERPRINT GENERATION + 70/15/15 SPLIT

from rdkit.Chem import AllChem
from rdkit.Chem.AtomPairs import Pairs, Torsions
from rdkit.Avalon import pyAvalonTools as AvalonTools
from rdkit.Chem.rdMHFPFingerprint import MHFPEncoder
from rdkit.Chem.rdFingerprintGenerator import GetAtomPairGenerator, GetTopologicalTorsionGenerator
from sklearn.model_selection import train_test_split
import numpy as np

print("=" * 60)
print("  FINGERPRINT GENERATION + 70/15/15 SPLIT")
print("=" * 60)

# Initialise MHFP encoder for SECFP
mhfp_encoder = MHFPEncoder(2048)

# Updated generators -- replaces deprecated methods
atom_pair_gen = GetAtomPairGenerator(fpSize=2048)
torsion_gen   = GetTopologicalTorsionGenerator(fpSize=2048)

# SMILES character set for sequence embedding
charset  = list("CNOSFPIHBrClcnosp#=@/\\1234567890()[]+-.")
char2idx = {c: i + 1 for i, c in enumerate(charset)}
MAX_LEN  = 200

def compute_fingerprints(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # 1. Avalon fingerprint (512 bits)
    avalon = np.array(
        AvalonTools.GetAvalonFP(mol, nBits=512),
        dtype=np.float32
    )

    # 2. Atom Pair fingerprint (2048 bits) -- updated generator
    atom_pair = np.array(
        atom_pair_gen.GetFingerprint(mol),
        dtype=np.float32
    )

    # 3. Topological Torsion fingerprint (2048 bits) -- updated generator
    torsion = np.array(
        torsion_gen.GetFingerprint(mol),
        dtype=np.float32
    )

    # 4. SECFP -- SMILES encoded circular fingerprint via MHFP (2048 bits)
    secfp = np.array(
        mhfp_encoder.EncodeMol(mol, radius=3, rings=True, isomeric=True),
        dtype=np.float32
    )

    # 5. Sequence-based embedding -- character level SMILES encoding (200 dims)
    seq = [char2idx.get(c, 0) for c in smiles[:MAX_LEN]]
    seq += [0] * (MAX_LEN - len(seq))
    seq_embed = np.array(seq, dtype=np.float32)

    # Concatenate all fingerprints into single feature vector
    return np.concatenate([avalon, atom_pair, torsion, secfp, seq_embed])

print("Computing fingerprints -- this may take a few minutes...")
fp_series = df["SMILES"].apply(compute_fingerprints)

# Identify and drop rows where fingerprint computation failed
valid_mask = fp_series.notna()
failed     = (~valid_mask).sum()
print(f"Failed fingerprint computations : {failed}")

if failed > 0:
    print(f"Dropping {failed} invalid rows...")
    df        = df[valid_mask].reset_index(drop=True)
    fp_series = fp_series[valid_mask].reset_index(drop=True)

# Build full fingerprint matrix
X = np.stack(fp_series.values).astype(np.float32)
y = df["Label"].values.astype(np.int64)

print(f"Fingerprint matrix shape        : {X.shape}")
print(f"Total molecules retained        : {len(df)}")

# Fingerprint composition summary
print("\n" + "=" * 60)
print("  FINGERPRINT COMPOSITION")
print("=" * 60)
fp_info = {
    "Avalon"              : 512,
    "Atom Pair"           : 2048,
    "Topological Torsion" : 2048,
    "SECFP (MHFP)"        : 2048,
    "Sequence Embedding"  : 200,
}
total_dims = 0
for name, dims in fp_info.items():
    print(f"{name:<25} : {dims} bits")
    total_dims += dims
print(f"{'Total feature vector':<25} : {total_dims} dimensions")

# Sparsity check
sparsity = (X == 0).mean() * 100
print(f"\nFingerprint matrix sparsity     : {sparsity:.2f}%")

# 70 / 15 / 15 SPLIT -- stratified by class
# Step 1 -- split off 70% train from 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size    = 0.30,
    random_state = 42,
    stratify     = y
)

# Step 2 -- split remaining 30% evenly into 15% val and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size    = 0.50,
    random_state = 42,
    stratify     = y_temp
)

total = len(y_train) + len(y_val) + len(y_test)

print("\n" + "=" * 60)
print("  70 / 15 / 15 SPLIT SUMMARY")
print("=" * 60)
print(f"Total molecules         : {total}")
print(f"Train  split            : {len(y_train)/total*100:.1f}%  ({len(y_train)} molecules)")
print(f"Val    split            : {len(y_val)/total*100:.1f}%  ({len(y_val)} molecules)")
print(f"Test   split            : {len(y_test)/total*100:.1f}%  ({len(y_test)} molecules)")

# Class distribution per split
print("\n" + "=" * 60)
print("  CLASS DISTRIBUTION PER SPLIT")
print("=" * 60)
for name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    blockers     = y_split.sum()
    non_blockers = len(y_split) - blockers
    print(f"{name:<6} -- Blocker: {blockers:4d} ({blockers/len(y_split)*100:.1f}%)  "
          f"Non-Blocker: {non_blockers:4d} ({non_blockers/len(y_split)*100:.1f}%)")

print("\n" + "=" * 60)
print("  STEP 9 COMPLETE -- READY FOR MODEL BUILDING")
print("=" * 60)
print(f"X_train : {X_train.shape}  -->  model training")
print(f"X_val   : {X_val.shape}   -->  hyperparameter tuning")
print(f"X_test  : {X_test.shape}   -->  final held-out evaluation")
print(f"Class weights -- Non-Blocker: 2.6831   Blocker: 0.6145")

  FINGERPRINT GENERATION + 70/15/15 SPLIT
Computing fingerprints -- this may take a few minutes...
Failed fingerprint computations : 0
Fingerprint matrix shape        : (5731, 6856)
Total molecules retained        : 5731

  FINGERPRINT COMPOSITION
Avalon                    : 512 bits
Atom Pair                 : 2048 bits
Topological Torsion       : 2048 bits
SECFP (MHFP)              : 2048 bits
Sequence Embedding        : 200 bits
Total feature vector      : 6856 dimensions

Fingerprint matrix sparsity     : 61.50%

  70 / 15 / 15 SPLIT SUMMARY
Total molecules         : 5731
Train  split            : 70.0%  (4011 molecules)
Val    split            : 15.0%  (860 molecules)
Test   split            : 15.0%  (860 molecules)

  CLASS DISTRIBUTION PER SPLIT
Train  -- Blocker: 3264 (81.4%)  Non-Blocker:  747 (18.6%)
Val    -- Blocker:  700 (81.4%)  Non-Blocker:  160 (18.6%)
Test   -- Blocker:  699 (81.3%)  Non-Blocker:  161 (18.7%)

  STEP 9 COMPLETE -- READY FOR MODEL BUILDING
X_train : (40

In [13]:
# STEP 10 · GNN GRAPH CONSTRUCTION
# Full atom features: extended + electronegativity, van der Waals radius, partial charge

import torch
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, rdPartialCharges
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader
from sklearn.model_selection import train_test_split

print("=" * 60)
print("  GNN GRAPH CONSTRUCTION -- FULL ATOM FEATURES")
print("=" * 60)

# ── ATOM FEATURE CONSTANTS ────────────────────────────────────

ATOM_TYPES = [
    "C", "N", "O", "S", "F", "P", "Cl", "Br", "I", "H", "B", "Si", "Unknown"
]

HYBRIDIZATION_TYPES = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
    Chem.rdchem.HybridizationType.UNSPECIFIED,
]

# Van der Waals radii (Angstroms) per element
VDW_RADII = {
    "H" : 1.20, "C" : 1.70, "N" : 1.55, "O" : 1.52,
    "F" : 1.47, "P" : 1.80, "S" : 1.80, "Cl": 1.75,
    "Br": 1.85, "I" : 1.98, "B" : 1.92, "Si": 2.10,
}

# Electronegativity (Pauling scale) per element
ELECTRONEGATIVITY = {
    "H" : 2.20, "C" : 2.55, "N" : 3.04, "O" : 3.44,
    "F" : 3.98, "P" : 2.19, "S" : 2.58, "Cl": 3.16,
    "Br": 2.96, "I" : 2.66, "B" : 2.04, "Si": 1.90,
}

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

def one_hot(value, choices):
    encoding = [0] * (len(choices) + 1)
    try:
        idx = choices.index(value)
    except ValueError:
        idx = len(choices)          # unknown category
    encoding[idx] = 1
    return encoding

def get_atom_features(atom, mol):
    symbol = atom.GetSymbol()

    # One-hot atom type (13 + 1 unknown = 14)
    atom_type_oh    = one_hot(symbol, ATOM_TYPES)

    # One-hot hybridization (6 + 1 = 7)
    hybrid_oh       = one_hot(atom.GetHybridization(), HYBRIDIZATION_TYPES)

    # One-hot degree capped at 6 (7)
    degree_oh       = one_hot(min(atom.GetDegree(), 6), list(range(7)))

    # One-hot formal charge capped range -2 to +2 (5)
    charge_oh       = one_hot(atom.GetFormalCharge(), [-2, -1, 0, 1, 2])

    # One-hot number of Hs (5)
    num_h_oh        = one_hot(
        min(atom.GetTotalNumHs(), 4), [0, 1, 2, 3, 4]
    )

    # Scalar features
    is_aromatic     = [int(atom.GetIsAromatic())]
    in_ring         = [int(atom.IsInRing())]
    in_ring3        = [int(atom.IsInRingSize(3))]
    in_ring4        = [int(atom.IsInRingSize(4))]
    in_ring5        = [int(atom.IsInRingSize(5))]
    in_ring6        = [int(atom.IsInRingSize(6))]

    # Chirality
    chiral_oh       = one_hot(
        str(atom.GetChiralTag()),
        ["CHI_UNSPECIFIED", "CHI_TETRAHEDRAL_CW",
         "CHI_TETRAHEDRAL_CCW", "CHI_OTHER"]
    )

    # Electronegativity (normalised 0-1 by max 3.98)
    electro         = [ELECTRONEGATIVITY.get(symbol, 2.20) / 3.98]

    # Van der Waals radius (normalised 0-1 by max 2.10)
    vdw             = [VDW_RADII.get(symbol, 1.70) / 2.10]

    # Gasteiger partial charge (computed per molecule)
    try:
        partial_q   = [float(atom.GetDoubleProp("_GasteigerCharge"))]
        if np.isnan(partial_q[0]) or np.isinf(partial_q[0]):
            partial_q = [0.0]
    except KeyError:
        partial_q   = [0.0]

    # Concatenate all features
    return (atom_type_oh + hybrid_oh + degree_oh + charge_oh +
            num_h_oh + is_aromatic + in_ring + in_ring3 +
            in_ring4 + in_ring5 + in_ring6 + chiral_oh +
            electro + vdw + partial_q)

def get_bond_features(bond):
    # One-hot bond type (4 + 1 = 5)
    bond_type_oh    = one_hot(bond.GetBondType(), BOND_TYPES)
    is_conjugated   = [int(bond.GetIsConjugated())]
    is_in_ring      = [int(bond.IsInRing())]
    is_aromatic     = [int(bond.GetIsAromatic())]
    stereo_oh       = one_hot(
        str(bond.GetStereo()),
        ["STEREONONE", "STEREOANY", "STEREOZ", "STEREOE"]
    )
    return bond_type_oh + is_conjugated + is_in_ring + is_aromatic + stereo_oh

def smiles_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Compute Gasteiger charges before feature extraction
    try:
        AllChem.ComputeGasteigerCharges(mol)
    except Exception:
        pass

    # Node features
    atom_features = [get_atom_features(atom, mol) for atom in mol.GetAtoms()]
    x = torch.tensor(atom_features, dtype=torch.float)

    # Edge index and edge features (bidirectional)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bf = get_bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr  += [bf, bf]

    if len(edge_index) == 0:
        # Single atom molecule -- self loop
        edge_index = [[0, 0]]
        edge_attr  = [get_bond_features(mol.GetBondWithIdx(0))
                      if mol.GetNumBonds() > 0
                      else [0] * 13]

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attr,  dtype=torch.float)

    return Data(
        x          = x,
        edge_index = edge_index,
        edge_attr  = edge_attr,
        y          = torch.tensor([label], dtype=torch.long),
        smiles     = smiles
    )

# ── BUILD GRAPH DATASET ───────────────────────────────────────

print("Building molecular graphs...")

graph_data = []
failed     = 0

for smiles, label in zip(df["SMILES"].values, df["Label"].values):
    g = smiles_to_graph(smiles, label)
    if g is not None:
        graph_data.append(g)
    else:
        failed += 1

print(f"Graphs constructed      : {len(graph_data)}")
print(f"Failed constructions    : {failed}")
print(f"Node feature dimension  : {graph_data[0].x.shape[1]}")
print(f"Edge feature dimension  : {graph_data[0].edge_attr.shape[1]}")

# ── 70 / 15 / 15 SPLIT -- CONSISTENT WITH FINGERPRINT SPLIT ──

indices = list(range(len(graph_data)))
labels  = [g.y.item() for g in graph_data]

idx_train, idx_temp, _, y_temp = train_test_split(
    indices, labels,
    test_size    = 0.30,
    random_state = 42,
    stratify     = labels
)

idx_val, idx_test, _, _ = train_test_split(
    idx_temp, y_temp,
    test_size    = 0.50,
    random_state = 42,
    stratify     = y_temp
)

train_graphs = [graph_data[i] for i in idx_train]
val_graphs   = [graph_data[i] for i in idx_val]
test_graphs  = [graph_data[i] for i in idx_test]

# ── DATALOADERS ───────────────────────────────────────────────

BATCH_SIZE = 64

train_loader = GeoDataLoader(train_graphs, batch_size=BATCH_SIZE,
                             shuffle=True,  drop_last=True)
val_loader   = GeoDataLoader(val_graphs,   batch_size=BATCH_SIZE,
                             shuffle=False, drop_last=False)
test_loader  = GeoDataLoader(test_graphs,  batch_size=BATCH_SIZE,
                             shuffle=False, drop_last=False)

# ── SUMMARY ───────────────────────────────────────────────────

print("\n" + "=" * 60)
print("  GRAPH DATASET SPLIT SUMMARY")
print("=" * 60)
print(f"Train graphs            : {len(train_graphs)}")
print(f"Val   graphs            : {len(val_graphs)}")
print(f"Test  graphs            : {len(test_graphs)}")

print("\n" + "=" * 60)
print("  FEATURE DIMENSIONS")
print("=" * 60)
sample = graph_data[0]
print(f"Node features (per atom): {sample.x.shape[1]}")
print(f"Edge features (per bond): {sample.edge_attr.shape[1]}")

node_dim_breakdown = {
    "Atom type one-hot"     : 14,
    "Hybridization one-hot" : 7,
    "Degree one-hot"        : 7,
    "Formal charge one-hot" : 5,
    "Num H one-hot"         : 5,
    "Aromaticity"           : 1,
    "Ring membership"       : 5,
    "Chirality one-hot"     : 5,
    "Electronegativity"     : 1,
    "Van der Waals radius"  : 1,
    "Partial charge"        : 1,
}
total_node_dims = 0
for feat, dims in node_dim_breakdown.items():
    print(f"  {feat:<28} : {dims}")
    total_node_dims += dims
print(f"  {'Total':<28} : {total_node_dims}")

print("\n" + "=" * 60)
print("  STEP 10 COMPLETE -- READY FOR MODEL BUILDING")
print("=" * 60)
print(f"train_loader, val_loader, test_loader ready")
print(f"Batch size              : {BATCH_SIZE}")
print(f"Node feature dim        : {sample.x.shape[1]}  --> GNN input_dim")
print(f"Edge feature dim        : {sample.edge_attr.shape[1]}  --> GNN edge_dim")

  GNN GRAPH CONSTRUCTION -- FULL ATOM FEATURES
Building molecular graphs...
Graphs constructed      : 5731
Failed constructions    : 0
Node feature dimension  : 55
Edge feature dimension  : 13

  GRAPH DATASET SPLIT SUMMARY
Train graphs            : 4011
Val   graphs            : 860
Test  graphs            : 860

  FEATURE DIMENSIONS
Node features (per atom): 55
Edge features (per bond): 13
  Atom type one-hot            : 14
  Hybridization one-hot        : 7
  Degree one-hot               : 7
  Formal charge one-hot        : 5
  Num H one-hot                : 5
  Aromaticity                  : 1
  Ring membership              : 5
  Chirality one-hot            : 5
  Electronegativity            : 1
  Van der Waals radius         : 1
  Partial charge               : 1
  Total                        : 52

  STEP 10 COMPLETE -- READY FOR MODEL BUILDING
train_loader, val_loader, test_loader ready
Batch size              : 64
Node feature dim        : 55  --> GNN input_dim
Edge feature d

In [15]:
# STEP 11 · CNN MODULE -- COMPLETE + ALL FEATURES
# Version 1 -- CNN with Attention
# Version 2 -- CNN without Attention
# Version 3 -- CNN with Fingerprint Features
# Includes  -- 3D/4D GradCAM, Saliency, t-SNE, Calibration,
#              Confusion Matrix, Bootstrap CI, Y-Randomisation,
#              Applicability Domain, Threshold Analysis,
#              Error Analysis, Filter Visualisation

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    f1_score, precision_score, recall_score,
    brier_score_loss, mean_squared_error, mean_absolute_error,
    roc_curve, precision_recall_curve, confusion_matrix,
)
from sklearn.calibration import calibration_curve
from sklearn.manifold import TSNE
from sklearn.utils import resample
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings("ignore")

print("=" * 60)
print("  CNN MODULE -- COMPLETE WITH ALL FEATURES")
print("=" * 60)

# ── CONSTANTS ─────────────────────────────────────────────────
INPUT_DIM     = X_train.shape[1]
HIDDEN_DIM    = 512
LATENT_DIM    = 256
NUM_CLASSES   = 2
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_WEIGHTS = torch.tensor([2.6831, 0.6145], dtype=torch.float).to(DEVICE)

print(f"Device                  : {DEVICE}")
print(f"Input dimension         : {INPUT_DIM}")
print(f"Class weights           : Non-Blocker=2.6831  Blocker=0.6145")

# ── PYTORCH DATALOADERS ───────────────────────────────────────
def make_loader(X, y, batch_size=64, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    ds  = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_fp_loader = make_loader(X_train, y_train, shuffle=True)
val_fp_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_fp_loader  = make_loader(X_test,  y_test,  shuffle=False)

# ── CHANNEL ATTENTION ─────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, num_channels, reduction=16):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc  = nn.Sequential(
            nn.Linear(num_channels, max(num_channels // reduction, 8)),
            nn.ReLU(),
            nn.Linear(max(num_channels // reduction, 8), num_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        w = self.gap(x).squeeze(-1)
        w = self.fc(w).unsqueeze(-1)
        return x * w

# ── VERSION 1 -- CNN WITH ATTENTION ───────────────────────────
class CNN_Attention(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_classes):
        super().__init__()
        self.input_proj  = nn.Linear(input_dim, hidden_dim)
        self.conv1       = nn.Conv1d(1,   64,  kernel_size=7, padding=3)
        self.bn1         = nn.BatchNorm1d(64)
        self.conv2       = nn.Conv1d(64,  128, kernel_size=5, padding=2)
        self.bn2         = nn.BatchNorm1d(128)
        self.conv3       = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3         = nn.BatchNorm1d(256)
        self.attention   = ChannelAttention(256)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.latent_proj = nn.Sequential(
            nn.Linear(256, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier  = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        x = self.input_proj(x).unsqueeze(1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.attention(x)
        x = self.global_pool(x).squeeze(-1)
        z = self.latent_proj(x)
        return self.classifier(z), z

    def get_gradcam_target(self):
        return self.conv3

    def get_conv_filters(self):
        return self.conv1, self.conv2, self.conv3

# ── VERSION 2 -- CNN WITHOUT ATTENTION ────────────────────────
class CNN_NoAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_classes):
        super().__init__()
        self.input_proj  = nn.Linear(input_dim, hidden_dim)
        self.conv1       = nn.Conv1d(1,   64,  kernel_size=7, padding=3)
        self.bn1         = nn.BatchNorm1d(64)
        self.conv2       = nn.Conv1d(64,  128, kernel_size=5, padding=2)
        self.bn2         = nn.BatchNorm1d(128)
        self.conv3       = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3         = nn.BatchNorm1d(256)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.latent_proj = nn.Sequential(
            nn.Linear(256, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier  = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        x = self.input_proj(x).unsqueeze(1)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.global_pool(x).squeeze(-1)
        z = self.latent_proj(x)
        return self.classifier(z), z

    def get_gradcam_target(self):
        return self.conv3

    def get_conv_filters(self):
        return self.conv1, self.conv2, self.conv3

# ── VERSION 3 -- CNN WITH FINGERPRINT FUSION ──────────────────
class CNN_Fingerprint(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_classes):
        super().__init__()
        self.input_proj  = nn.Linear(input_dim, hidden_dim)
        self.conv1       = nn.Conv1d(1,   64,  kernel_size=7, padding=3)
        self.bn1         = nn.BatchNorm1d(64)
        self.conv2       = nn.Conv1d(64,  128, kernel_size=5, padding=2)
        self.bn2         = nn.BatchNorm1d(128)
        self.conv3       = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn3         = nn.BatchNorm1d(256)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fp_branch   = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.fusion      = nn.Sequential(
            nn.Linear(256 + latent_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier  = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        fp_feat = self.fp_branch(x)
        x_conv  = self.input_proj(x).unsqueeze(1)
        x_conv  = F.relu(self.bn1(self.conv1(x_conv)))
        x_conv  = F.relu(self.bn2(self.conv2(x_conv)))
        x_conv  = F.relu(self.bn3(self.conv3(x_conv)))
        x_conv  = self.global_pool(x_conv).squeeze(-1)
        z       = self.fusion(torch.cat([x_conv, fp_feat], dim=1))
        return self.classifier(z), z

    def get_gradcam_target(self):
        return self.conv3

    def get_conv_filters(self):
        return self.conv1, self.conv2, self.conv3

# ── GRADCAM ───────────────────────────────────────────────────
class GradCAM1D:
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def compute(self, input_tensor, target_class):
        self.model.eval()
        input_tensor = input_tensor.to(DEVICE).requires_grad_(True)
        logits, _    = self.model(input_tensor)
        self.model.zero_grad()
        logits[0, target_class].backward()
        weights = self.gradients.mean(dim=2, keepdim=True)
        cam     = F.relu((weights * self.activations).sum(dim=1)).squeeze(0)
        cam     = cam.cpu().numpy()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam

# ── SALIENCY MAP ──────────────────────────────────────────────
def compute_saliency(model, input_tensor, target_class):
    model.eval()
    x = input_tensor.to(DEVICE).requires_grad_(True)
    logits, _ = model(x)
    model.zero_grad()
    logits[0, target_class].backward()
    saliency = x.grad.abs().squeeze(0).cpu().numpy()
    if saliency.max() > 0:
        saliency = saliency / saliency.max()
    return saliency

# ── TRAINING FUNCTION -- ALL METRICS TRACKED ──────────────────
def train_model(model, train_loader, val_loader, epochs, lr, weight_decay):
    model     = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode="max", patience=5, factor=0.5)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

    history  = {
        "train_loss" : [], "val_loss"  : [],
        "train_auc"  : [], "val_auc"   : [],
        "train_acc"  : [], "val_acc"   : [],
        "train_f1"   : [], "val_f1"    : [],
        "train_prec" : [], "val_prec"  : [],
        "train_rec"  : [], "val_rec"   : [],
        "train_brier": [], "val_brier" : [],
        "train_rmse" : [], "val_rmse"  : [],
        "train_mae"  : [], "val_mae"   : [],
    }

    best_val_auc = 0.0
    best_state   = None

    for epoch in range(epochs):
        model.train()
        t_losses, t_probs, t_preds, t_labels = [], [], [], []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(X_batch)
            loss      = criterion(logits, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            t_losses.append(loss.item())
            t_probs.extend(
                F.softmax(logits, dim=1)[:,1].detach().cpu().numpy())
            t_preds.extend(
                logits.argmax(dim=1).detach().cpu().numpy())
            t_labels.extend(y_batch.cpu().numpy())

        model.eval()
        v_losses, v_probs, v_preds, v_labels = [], [], [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                logits, _ = model(X_batch)
                loss      = criterion(logits, y_batch)
                v_losses.append(loss.item())
                v_probs.extend(
                    F.softmax(logits, dim=1)[:,1].cpu().numpy())
                v_preds.extend(
                    logits.argmax(dim=1).cpu().numpy())
                v_labels.extend(y_batch.cpu().numpy())

        tp, tl, td = np.array(t_probs), np.array(t_labels), np.array(t_preds)
        vp, vl, vd = np.array(v_probs), np.array(v_labels), np.array(v_preds)

        history["train_loss"].append(np.mean(t_losses))
        history["val_loss"].append(np.mean(v_losses))
        history["train_auc"].append(roc_auc_score(tl, tp))
        history["val_auc"].append(roc_auc_score(vl, vp))
        history["train_acc"].append(accuracy_score(tl, td))
        history["val_acc"].append(accuracy_score(vl, vd))
        history["train_f1"].append(f1_score(tl, td, zero_division=0))
        history["val_f1"].append(f1_score(vl, vd, zero_division=0))
        history["train_prec"].append(precision_score(tl, td, zero_division=0))
        history["val_prec"].append(precision_score(vl, vd, zero_division=0))
        history["train_rec"].append(recall_score(tl, td, zero_division=0))
        history["val_rec"].append(recall_score(vl, vd, zero_division=0))
        history["train_brier"].append(brier_score_loss(tl, tp))
        history["val_brier"].append(brier_score_loss(vl, vp))
        history["train_rmse"].append(np.sqrt(mean_squared_error(tl, tp)))
        history["val_rmse"].append(np.sqrt(mean_squared_error(vl, vp)))
        history["train_mae"].append(mean_absolute_error(tl, tp))
        history["val_mae"].append(mean_absolute_error(vl, vp))

        scheduler.step(history["val_auc"][-1])

        if history["val_auc"][-1] > best_val_auc:
            best_val_auc = history["val_auc"][-1]
            best_state   = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:03d} | "
                  f"Train Loss: {history['train_loss'][-1]:.4f}  "
                  f"AUC: {history['train_auc'][-1]:.4f}  "
                  f"Acc: {history['train_acc'][-1]:.4f} | "
                  f"Val Loss: {history['val_loss'][-1]:.4f}  "
                  f"AUC: {history['val_auc'][-1]:.4f}  "
                  f"Acc: {history['val_acc'][-1]:.4f}")

    model.load_state_dict(best_state)
    return model, history

# ── EVALUATION FUNCTION ───────────────────────────────────────
def evaluate_model(model, loader):
    model.eval()
    all_probs, all_preds, all_labels, all_latents = [], [], [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch   = X_batch.to(DEVICE)
            logits, z = model(X_batch)
            probs     = F.softmax(logits, dim=1)[:,1].cpu().numpy()
            preds     = logits.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())
            all_latents.append(z.cpu().numpy())

    p    = np.array(all_probs)
    y    = np.array(all_labels)
    pred = np.array(all_preds)
    Z    = np.concatenate(all_latents, axis=0)

    return {
        "ROC-AUC"   : roc_auc_score(y, p),
        "PR-AUC"    : average_precision_score(y, p),
        "Accuracy"  : accuracy_score(y, pred),
        "F1"        : f1_score(y, pred),
        "Precision" : precision_score(y, pred),
        "Recall"    : recall_score(y, pred),
        "Brier"     : brier_score_loss(y, p),
        "RMSE"      : np.sqrt(mean_squared_error(y, p)),
        "MAE"       : mean_absolute_error(y, p),
        "probs"     : p,
        "labels"    : y,
        "preds"     : pred,
        "latents"   : Z,
    }

# ── BOOTSTRAP CONFIDENCE INTERVALS ───────────────────────────
def bootstrap_ci(y_true, y_prob, metric_fn, n_boot=1000, ci=95):
    scores = []
    for _ in range(n_boot):
        idx   = resample(np.arange(len(y_true)), random_state=None)
        try:
            scores.append(metric_fn(y_true[idx], y_prob[idx]))
        except Exception:
            pass
    lower = np.percentile(scores, (100 - ci) / 2)
    upper = np.percentile(scores, 100 - (100 - ci) / 2)
    return np.mean(scores), lower, upper

# ── Y-RANDOMISATION TEST ──────────────────────────────────────
def y_randomisation_test(model_class, train_loader, val_loader,
                          best_config, n_runs=5):
    rand_aucs = []
    print("  Running Y-Randomisation test...")
    for run in range(n_runs):
        rand_model = model_class(INPUT_DIM, HIDDEN_DIM, LATENT_DIM, NUM_CLASSES)

        # Shuffle labels in a copy of train loader
        X_rand = X_train.copy()
        y_rand = y_train.copy()
        np.random.shuffle(y_rand)
        rand_loader = make_loader(X_rand, y_rand, shuffle=True)

        rand_model, _ = train_model(
            rand_model, rand_loader, val_fp_loader,
            epochs       = 20,
            lr           = best_config["lr"],
            weight_decay = best_config["weight_decay"]
        )
        metrics   = evaluate_model(rand_model, val_fp_loader)
        rand_aucs.append(metrics["ROC-AUC"])
        print(f"    Run {run+1}: Val ROC-AUC = {metrics['ROC-AUC']:.4f}")

    return np.array(rand_aucs)

# ── APPLICABILITY DOMAIN ──────────────────────────────────────
def applicability_domain(X_train_ref, X_test_ref, threshold_pct=95):
    from sklearn.metrics.pairwise import cosine_similarity
    # Compute mean cosine similarity of each test molecule to training set
    sim_matrix  = cosine_similarity(X_test_ref, X_train_ref)
    mean_sim    = sim_matrix.max(axis=1)  # nearest neighbour similarity
    threshold   = np.percentile(
        cosine_similarity(X_train_ref, X_train_ref).mean(axis=1),
        100 - threshold_pct
    )
    in_domain   = mean_sim >= threshold
    return mean_sim, threshold, in_domain

# ── MANUAL GRID SEARCH ────────────────────────────────────────
GRID = [
    {"lr": 1e-3, "weight_decay": 1e-4, "epochs": 50},
    {"lr": 5e-4, "weight_decay": 1e-4, "epochs": 50},
    {"lr": 1e-4, "weight_decay": 1e-5, "epochs": 50},
]

CNN_VERSIONS = {
    "CNN_Attention"   : CNN_Attention,
    "CNN_NoAttention" : CNN_NoAttention,
    "CNN_Fingerprint" : CNN_Fingerprint,
}

all_results    = {}
all_histories  = {}
best_models    = {}
best_configs   = {}

for version_name, ModelClass in CNN_VERSIONS.items():
    print("\n" + "=" * 60)
    print(f"  TRAINING : {version_name}")
    print("=" * 60)

    best_auc     = 0.0
    best_config  = None
    best_model_v = None
    best_history = None

    for config in GRID:
        print(f"\n  Config: lr={config['lr']}  "
              f"wd={config['weight_decay']}  "
              f"epochs={config['epochs']}")
        model = ModelClass(INPUT_DIM, HIDDEN_DIM, LATENT_DIM, NUM_CLASSES)
        model, history = train_model(
            model, train_fp_loader, val_fp_loader,
            epochs       = config["epochs"],
            lr           = config["lr"],
            weight_decay = config["weight_decay"]
        )
        val_metrics = evaluate_model(model, val_fp_loader)
        print(f"  Val ROC-AUC : {val_metrics['ROC-AUC']:.4f}")

        if val_metrics["ROC-AUC"] > best_auc:
            best_auc     = val_metrics["ROC-AUC"]
            best_config  = config
            best_model_v = model
            best_history = history

    best_models[version_name]   = best_model_v
    all_histories[version_name] = best_history
    best_configs[version_name]  = best_config
    test_metrics                = evaluate_model(best_model_v, test_fp_loader)
    all_results[version_name]   = test_metrics

    print(f"\n  Best config    : {best_config}")
    print(f"  Test ROC-AUC   : {test_metrics['ROC-AUC']:.4f}")
    print(f"  Test PR-AUC    : {test_metrics['PR-AUC']:.4f}")
    print(f"  Test Accuracy  : {test_metrics['Accuracy']:.4f}")
    print(f"  Test F1        : {test_metrics['F1']:.4f}")
    print(f"  Test Precision : {test_metrics['Precision']:.4f}")
    print(f"  Test Recall    : {test_metrics['Recall']:.4f}")
    print(f"  Test Brier     : {test_metrics['Brier']:.4f}")
    print(f"  Test RMSE      : {test_metrics['RMSE']:.4f}")
    print(f"  Test MAE       : {test_metrics['MAE']:.4f}")

# ── COLOURS ───────────────────────────────────────────────────
COLORS = {
    "CNN_Attention"   : "#2196F3",
    "CNN_NoAttention" : "#FF5722",
    "CNN_Fingerprint" : "#4CAF50",
}
versions = list(all_histories.keys())

# =============================================================
# VISUALISATION BLOCK 1 -- TRAINING CURVES
# =============================================================

# Figure 1 -- Loss + Accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (t_key, v_key, title, ylabel) in zip(axes, [
    ("train_loss", "val_loss", "Loss Curves",     "Loss"),
    ("train_acc",  "val_acc",  "Accuracy Curves", "Accuracy"),
]):
    for name in versions:
        ax.plot(all_histories[name][t_key], color=COLORS[name],
                linestyle="--", alpha=0.5, linewidth=1.2,
                label=f"{name} Train")
        ax.plot(all_histories[name][v_key], color=COLORS[name],
                linestyle="-", linewidth=2, label=f"{name} Val")
    if "acc" in t_key:
        ax.axhline(0.87, color="red", linestyle="--",
                   linewidth=1.5, label="Target (0.87)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle("CNN -- Loss & Accuracy Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_loss_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 2 -- ROC-AUC + F1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (t_key, v_key, title, ylabel) in zip(axes, [
    ("train_auc", "val_auc", "ROC-AUC Curves", "ROC-AUC"),
    ("train_f1",  "val_f1",  "F1 Curves",      "F1 Score"),
]):
    for name in versions:
        ax.plot(all_histories[name][t_key], color=COLORS[name],
                linestyle="--", alpha=0.5, linewidth=1.2,
                label=f"{name} Train")
        ax.plot(all_histories[name][v_key], color=COLORS[name],
                linestyle="-", linewidth=2, label=f"{name} Val")
    ax.axhline(0.87, color="red", linestyle="--",
               linewidth=1.5, label="Target (0.87)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle("CNN -- ROC-AUC & F1 Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_auc_f1.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 3 -- Precision + Recall
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (t_key, v_key, title, ylabel) in zip(axes, [
    ("train_prec", "val_prec", "Precision Curves", "Precision"),
    ("train_rec",  "val_rec",  "Recall Curves",    "Recall"),
]):
    for name in versions:
        ax.plot(all_histories[name][t_key], color=COLORS[name],
                linestyle="--", alpha=0.5, linewidth=1.2,
                label=f"{name} Train")
        ax.plot(all_histories[name][v_key], color=COLORS[name],
                linestyle="-", linewidth=2, label=f"{name} Val")
    ax.axhline(0.87, color="red", linestyle="--",
               linewidth=1.5, label="Target (0.87)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle("CNN -- Precision & Recall Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_precision_recall_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 4 -- Brier + RMSE + MAE
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (t_key, v_key, title) in zip(axes, [
    ("train_brier", "val_brier", "Brier Score"),
    ("train_rmse",  "val_rmse",  "RMSE"),
    ("train_mae",   "val_mae",   "MAE"),
]):
    for name in versions:
        ax.plot(all_histories[name][t_key], color=COLORS[name],
                linestyle="--", alpha=0.5, linewidth=1.2,
                label=f"{name} Train")
        ax.plot(all_histories[name][v_key], color=COLORS[name],
                linestyle="-", linewidth=2, label=f"{name} Val")
    ax.set_title(f"{title} (lower is better)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle("CNN -- Calibration Metric Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_calibration_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 2 -- TEST SET CURVES
# =============================================================

# Figure 5 -- ROC + PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for name, res in all_results.items():
    fpr, tpr, _ = roc_curve(res["labels"], res["probs"])
    optimal_idx = np.argmax(tpr - fpr)
    ax.plot(fpr, tpr, color=COLORS[name], linewidth=2.5,
            label=f"{name} (AUC={res['ROC-AUC']:.4f})")
    ax.scatter(fpr[optimal_idx], tpr[optimal_idx],
               color=COLORS[name], s=100, zorder=5, marker="*")
    ax.fill_between(fpr, tpr, alpha=0.05, color=COLORS[name])
ax.plot([0,1],[0,1], "k--", alpha=0.4, label="Random")
ax.set_title("ROC Curves -- Test Set", fontsize=12, fontweight="bold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[1]
for name, res in all_results.items():
    prec, rec, _ = precision_recall_curve(res["labels"], res["probs"])
    ax.plot(rec, prec, color=COLORS[name], linewidth=2.5,
            label=f"{name} (PR-AUC={res['PR-AUC']:.4f})")
    ax.fill_between(rec, prec, alpha=0.05, color=COLORS[name])
baseline = list(all_results.values())[0]["labels"].mean()
ax.axhline(baseline, color="red", linestyle="--",
           linewidth=1.5, label=f"Baseline ({baseline:.2f})")
ax.set_title("PR Curves -- Test Set", fontsize=12, fontweight="bold")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
plt.suptitle("CNN -- Test Set Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_test_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 6 -- All 9 metrics bar chart
all_metric_cfg = {
    "ROC-AUC"   : {"higher": True,  "target": 0.87},
    "PR-AUC"    : {"higher": True,  "target": 0.87},
    "Accuracy"  : {"higher": True,  "target": 0.87},
    "F1"        : {"higher": True,  "target": 0.87},
    "Precision" : {"higher": True,  "target": 0.87},
    "Recall"    : {"higher": True,  "target": 0.87},
    "Brier"     : {"higher": False, "target": 0.15},
    "RMSE"      : {"higher": False, "target": 0.35},
    "MAE"       : {"higher": False, "target": 0.25},
}
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes_flat = axes.flatten()
x, width  = np.arange(len(versions)), 0.5
for i, (metric, info) in enumerate(all_metric_cfg.items()):
    ax   = axes_flat[i]
    vals = [all_results[v][metric] for v in versions]
    bars = ax.bar(x, vals, width,
                  color=[COLORS[v] for v in versions],
                  edgecolor="black", alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{val:.4f}", ha="center",
                va="bottom", fontsize=9, fontweight="bold")
    direction = "higher" if info["higher"] else "lower"
    ax.axhline(info["target"], color="red", linestyle="--",
               linewidth=1.5,
               label=f"Target ({info['target']}) -- {direction} is better")
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([v.replace("CNN_", "") for v in versions], fontsize=9)
    ax.set_ylabel(metric)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_ylim(0, max(vals) * 1.25 if not info["higher"] else 1.12)
plt.suptitle("CNN -- All 9 Metrics (Test Set)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_all_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 3 -- CONFUSION MATRICES
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, all_results.items()):
    cm  = confusion_matrix(res["labels"], res["preds"])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im  = ax.imshow(cm_norm, cmap="gray_r", vmin=0, vmax=1)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Non-Blocker", "Blocker"], fontsize=9)
    ax.set_yticklabels(["Non-Blocker", "Blocker"], fontsize=9)
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("Actual",    fontsize=10)
    ax.set_title(f"Confusion Matrix\n{name}", fontsize=11, fontweight="bold")
    for r in range(2):
        for c in range(2):
            ax.text(c, r,
                    f"{cm[r,c]}\n({cm_norm[r,c]:.2f})",
                    ha="center", va="center",
                    fontsize=11, fontweight="bold",
                    color="white" if cm_norm[r,c] > 0.5 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("CNN -- Confusion Matrices (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 4 -- CALIBRATION CURVES
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, all_results.items()):
    frac_pos, mean_pred = calibration_curve(
        res["labels"], res["probs"], n_bins=10)
    ax.plot(mean_pred, frac_pos, "s-", color=COLORS[name],
            linewidth=2, markersize=6, label=name)
    ax.plot([0,1],[0,1], "k--", alpha=0.5, label="Perfect calibration")
    ax.fill_between(mean_pred, frac_pos, mean_pred,
                    alpha=0.1, color=COLORS[name])
    ax.set_title(f"Calibration Curve\n{name}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Fraction of Positives")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
plt.suptitle("CNN -- Reliability Diagrams (Calibration)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_calibration_reliability.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 5 -- THRESHOLD ANALYSIS
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, all_results.items()):
    thresholds = np.linspace(0.1, 0.9, 50)
    accs, f1s, precs, recs = [], [], [], []
    for thr in thresholds:
        pred_thr = (res["probs"] >= thr).astype(int)
        accs.append(accuracy_score(res["labels"], pred_thr))
        f1s.append(f1_score(res["labels"], pred_thr, zero_division=0))
        precs.append(precision_score(res["labels"], pred_thr, zero_division=0))
        recs.append(recall_score(res["labels"], pred_thr, zero_division=0))

    ax.plot(thresholds, accs,  color="#2196F3", linewidth=2, label="Accuracy")
    ax.plot(thresholds, f1s,   color="#4CAF50", linewidth=2, label="F1")
    ax.plot(thresholds, precs, color="#FF5722", linewidth=2, label="Precision")
    ax.plot(thresholds, recs,  color="#9C27B0", linewidth=2, label="Recall")
    ax.axvline(0.5, color="black", linestyle="--",
               linewidth=1.5, label="Default threshold (0.5)")

    # Mark optimal F1 threshold
    best_thr = thresholds[np.argmax(f1s)]
    ax.axvline(best_thr, color="green", linestyle=":",
               linewidth=1.5, label=f"Optimal F1 threshold ({best_thr:.2f})")

    ax.set_title(f"Threshold Analysis\n{name}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Decision Threshold")
    ax.set_ylabel("Score")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.1, 0.9)
    ax.set_ylim(0, 1.05)
plt.suptitle("CNN -- Threshold Analysis (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 6 -- BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================
print("\n" + "=" * 60)
print("  BOOTSTRAP CONFIDENCE INTERVALS (1000 iterations)")
print("=" * 60)

bootstrap_results = {}
for name, res in all_results.items():
    print(f"\n  {name}")
    ci_dict = {}
    for metric, fn in [
        ("ROC-AUC",  lambda y, p: roc_auc_score(y, p)),
        ("PR-AUC",   lambda y, p: average_precision_score(y, p)),
        ("Accuracy", lambda y, p: accuracy_score(y, (p >= 0.5).astype(int))),
        ("F1",       lambda y, p: f1_score(y, (p >= 0.5).astype(int),
                                           zero_division=0)),
    ]:
        mean, lo, hi = bootstrap_ci(res["labels"], res["probs"], fn)
        ci_dict[metric] = (mean, lo, hi)
        print(f"    {metric:<12} : {mean:.4f}  "
              f"95% CI [{lo:.4f} -- {hi:.4f}]")
    bootstrap_results[name] = ci_dict

# Plot bootstrap CIs
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
ci_metrics = ["ROC-AUC", "PR-AUC", "Accuracy", "F1"]
for ax, metric in zip(axes, ci_metrics):
    means = [bootstrap_results[v][metric][0] for v in versions]
    lows  = [bootstrap_results[v][metric][1] for v in versions]
    highs = [bootstrap_results[v][metric][2] for v in versions]
    yerr  = [[m - l for m, l in zip(means, lows)],
              [h - m for m, h in zip(means, highs)]]
    bars  = ax.bar(np.arange(len(versions)), means, 0.5,
                   color=[COLORS[v] for v in versions],
                   edgecolor="black", alpha=0.85,
                   yerr=yerr, capsize=6, error_kw={"linewidth": 2})
    ax.axhline(0.87, color="red", linestyle="--",
               linewidth=1.5, label="Target (0.87)")
    ax.set_title(f"{metric}\n(95% Bootstrap CI)",
                 fontsize=11, fontweight="bold")
    ax.set_xticks(np.arange(len(versions)))
    ax.set_xticklabels([v.replace("CNN_", "") for v in versions],
                       fontsize=9, rotation=15)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.12)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, axis="y")
plt.suptitle("CNN -- Bootstrap Confidence Intervals (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_bootstrap_ci.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 7 -- t-SNE LATENT SPACE
# =============================================================
print("\n" + "=" * 60)
print("  t-SNE LATENT SPACE VISUALISATION")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, all_results.items()):
    print(f"  Computing t-SNE for {name}...")
    tsne = TSNE(n_components=2, random_state=42,
            perplexity=30, max_iter=1000)
    Z_2d   = tsne.fit_transform(res["latents"])

    # Plot by true label
    for lbl, label_name, color in [
        (0, "Non-Blocker", "#FF5722"),
        (1, "Blocker",     "#2196F3")
    ]:
        mask = res["labels"] == lbl
        ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                   c=color, s=8, alpha=0.6,
                   label=f"{label_name} (n={mask.sum()})")

    ax.set_title(f"t-SNE Latent Space\n{name}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("t-SNE Dimension 1")
    ax.set_ylabel("t-SNE Dimension 2")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle("CNN -- t-SNE Latent Space (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 8 -- PROBABILITY DISTRIBUTION
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, all_results.items()):
    blockers     = res["probs"][res["labels"] == 1]
    non_blockers = res["probs"][res["labels"] == 0]
    ax.hist(non_blockers, bins=40, alpha=0.6,
            color="#FF5722", label="Non-Blocker", edgecolor="black")
    ax.hist(blockers, bins=40, alpha=0.6,
            color="#2196F3", label="Blocker",     edgecolor="black")
    ax.axvline(0.5, color="black", linestyle="--",
               linewidth=1.5, label="Default boundary (0.5)")
    ax.set_title(f"Probability Distribution\n{name}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted Probability (Blocker)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle("CNN -- Prediction Confidence Distribution",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_probability_dist.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 9 -- ERROR ANALYSIS
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, all_results.items()):
    correct   = res["labels"] == res["preds"]
    incorrect = ~correct

    ax.scatter(
        np.where(correct)[0], res["probs"][correct],
        c="#4CAF50", s=8, alpha=0.5, label=f"Correct ({correct.sum()})"
    )
    ax.scatter(
        np.where(incorrect)[0], res["probs"][incorrect],
        c="#F44336", s=15, alpha=0.8,
        marker="x", label=f"Misclassified ({incorrect.sum()})"
    )
    ax.axhline(0.5, color="black", linestyle="--",
               linewidth=1.0, alpha=0.5)
    ax.set_title(f"Error Analysis\n{name}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Sample Index")
    ax.set_ylabel("Predicted Probability (Blocker)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle("CNN -- Error Analysis (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 10 -- APPLICABILITY DOMAIN
# =============================================================
print("\n" + "=" * 60)
print("  APPLICABILITY DOMAIN ANALYSIS")
print("=" * 60)

sim_scores, ad_threshold, in_domain = applicability_domain(X_train, X_test)
print(f"  AD threshold            : {ad_threshold:.4f}")
print(f"  Test molecules in domain: {in_domain.sum()} / {len(in_domain)} "
      f"({in_domain.mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(sim_scores[in_domain],
        bins=40, color="#4CAF50", alpha=0.7,
        edgecolor="black", label="In domain")
ax.hist(sim_scores[~in_domain],
        bins=40, color="#F44336", alpha=0.7,
        edgecolor="black", label="Out of domain")
ax.axvline(ad_threshold, color="black", linestyle="--",
           linewidth=2, label=f"Threshold ({ad_threshold:.3f})")
ax.set_title("Applicability Domain\nSimilarity Distribution",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Max Cosine Similarity to Training Set")
ax.set_ylabel("Count")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
best_name = max(all_results, key=lambda n: all_results[n]["ROC-AUC"])
res       = all_results[best_name]
ax.scatter(
    sim_scores[in_domain],
    res["probs"][in_domain],
    c="#4CAF50", s=10, alpha=0.5, label="In domain"
)
ax.scatter(
    sim_scores[~in_domain],
    res["probs"][~in_domain],
    c="#F44336", s=10, alpha=0.5, label="Out of domain"
)
ax.axvline(ad_threshold, color="black", linestyle="--",
           linewidth=1.5)
ax.set_title(f"AD vs Prediction Confidence\n{best_name}",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Similarity to Training Set")
ax.set_ylabel("Predicted Probability (Blocker)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle("CNN -- Applicability Domain Analysis",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("CNN_applicability_domain.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 11 -- Y-RANDOMISATION TEST
# =============================================================
print("\n" + "=" * 60)
print("  Y-RANDOMISATION TEST")
print("=" * 60)

rand_results = {}
for version_name, ModelClass in CNN_VERSIONS.items():
    print(f"\n  {version_name}")
    rand_aucs = y_randomisation_test(
        ModelClass, train_fp_loader, val_fp_loader,
        best_configs[version_name], n_runs=5
    )
    rand_results[version_name] = rand_aucs
    real_auc = all_results[version_name]["ROC-AUC"]
    print(f"  Real ROC-AUC            : {real_auc:.4f}")
    print(f"  Random mean ROC-AUC     : {rand_aucs.mean():.4f}")
    print(f"  Separation              : {real_auc - rand_aucs.mean():.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(versions))
for i, name in enumerate(versions):
    real_auc  = all_results[name]["ROC-AUC"]
    rand_aucs = rand_results[name]
    ax.bar(i - 0.2, real_auc, 0.35,
           color=COLORS[name], edgecolor="black",
           alpha=0.9, label=f"{name} Real")
    ax.bar(i + 0.2, rand_aucs.mean(), 0.35,
           color=COLORS[name], edgecolor="black",
           alpha=0.4, hatch="//",
           label=f"{name} Random")
    ax.errorbar(i + 0.2, rand_aucs.mean(),
                yerr=rand_aucs.std(),
                fmt="none", color="black", capsize=5)

ax.axhline(0.87, color="red", linestyle="--",
           linewidth=1.5, label="Target (0.87)")
ax.axhline(0.5, color="gray", linestyle=":",
           linewidth=1.0, label="Random chance (0.5)")
ax.set_xticks(x_pos)
ax.set_xticklabels([v.replace("CNN_", "") for v in versions], fontsize=10)
ax.set_ylabel("ROC-AUC")
ax.set_title("Y-Randomisation Test -- Real vs Random Labels",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("CNN_y_randomisation.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 12 -- GRADCAM 3D/4D + SALIENCY
# =============================================================
print("\n" + "=" * 60)
print("  GRADCAM + SALIENCY ANALYSIS")
print("=" * 60)

fp_segments = {
    "Avalon"   : (0,    512),
    "AtomPair" : (512,  2560),
    "Torsion"  : (2560, 4608),
    "SECFP"    : (4608, 6656),
    "SeqEmbed" : (6656, 6856),
}
segment_bounds = [0, 512, 2560, 4608, 6656, 6856]
segment_names  = list(fp_segments.keys())
segment_colors = ["#212121", "#616161", "#9E9E9E", "#CCCCCC", "#F5F5F5"]

blocker_idx    = np.where(y_test == 1)[0][:3]
nonblocker_idx = np.where(y_test == 0)[0][:3]
sample_idx     = np.concatenate([blocker_idx, nonblocker_idx])
X_sample       = X_test[sample_idx]
y_sample       = y_test[sample_idx]

for version_name, model in best_models.items():
    print(f"\n  GradCAM + Saliency for {version_name}...")
    target_layer = model.get_gradcam_target()
    gradcam      = GradCAM1D(model, target_layer)

    cam_maps  = []
    sal_maps  = []

    for i in range(len(sample_idx)):
        x_t  = torch.tensor(X_sample[i:i+1], dtype=torch.float32)
        cam  = gradcam.compute(x_t, target_class=int(y_sample[i]))
        sal  = compute_saliency(model,
                                torch.tensor(X_sample[i:i+1],
                                             dtype=torch.float32),
                                target_class=int(y_sample[i]))
        cam_maps.append(cam)
        sal_maps.append(sal)

    fig = plt.figure(figsize=(24, 22))
    fig.suptitle(
        f"GradCAM + Saliency Analysis -- {version_name}\n"
        f"hERG Cardiotoxicity Prediction",
        fontsize=14, fontweight="bold"
    )
    gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.55, wspace=0.4)

    # Row 1 -- 2D grayscale GradCAM
    for i in range(3):
        ax  = fig.add_subplot(gs[0, i])
        cam = cam_maps[i]
        grid_w  = 64
        padded  = np.zeros(grid_w * ((len(cam) // grid_w) + 1))
        padded[:len(cam)] = cam
        cam_2d  = padded[:grid_w * (len(padded) // grid_w)].reshape(-1, grid_w)
        ax.imshow(cam_2d, cmap="gray", aspect="auto", interpolation="bilinear")
        ax.set_title(
            f"GradCAM Sample {i+1}\n"
            f"{'Blocker' if y_sample[i]==1 else 'Non-Blocker'}",
            fontsize=9, fontweight="bold"
        )
        ax.set_xlabel("Feature Position", fontsize=7)
        ax.set_ylabel("Feature Block",    fontsize=7)
        ax.tick_params(labelsize=6)
        for b in segment_bounds[1:-1]:
            ax.axvline(b // grid_w, color="red", linewidth=0.8, alpha=0.7)

    # Row 2 -- 2D grayscale Saliency
    for i in range(3):
        ax  = fig.add_subplot(gs[1, i])
        sal = sal_maps[i]
        grid_w  = 64
        padded  = np.zeros(grid_w * ((len(sal) // grid_w) + 1))
        padded[:len(sal)] = sal
        sal_2d  = padded[:grid_w * (len(padded) // grid_w)].reshape(-1, grid_w)
        ax.imshow(sal_2d, cmap="gray", aspect="auto", interpolation="bilinear")
        ax.set_title(
            f"Saliency Sample {i+1}\n"
            f"{'Blocker' if y_sample[i]==1 else 'Non-Blocker'}",
            fontsize=9, fontweight="bold"
        )
        ax.set_xlabel("Feature Position", fontsize=7)
        ax.set_ylabel("Feature Block",    fontsize=7)
        ax.tick_params(labelsize=6)

    # Row 3 -- 3D surface GradCAM
    for i in range(3):
        ax  = fig.add_subplot(gs[2, i], projection="3d")
        cam = cam_maps[i]
        grid_w  = 64
        padded  = np.zeros(grid_w * ((len(cam) // grid_w) + 1))
        padded[:len(cam)] = cam
        cam_2d  = padded[:grid_w * (len(padded) // grid_w)].reshape(-1, grid_w)
        X_grid  = np.arange(cam_2d.shape[1])
        Y_grid  = np.arange(cam_2d.shape[0])
        X_mesh, Y_mesh = np.meshgrid(X_grid, Y_grid)
        ax.plot_surface(X_mesh, Y_mesh, cam_2d,
                        cmap="gray", edgecolor="none", alpha=0.9)
        ax.set_title(
            f"3D GradCAM\n{'Blocker' if y_sample[i]==1 else 'Non-Blocker'}",
            fontsize=9, fontweight="bold"
        )
        ax.set_xlabel("Feature\nPosition", fontsize=6)
        ax.set_ylabel("Feature\nBlock",    fontsize=6)
        ax.set_zlabel("Activation",        fontsize=6)
        ax.tick_params(labelsize=5)
        ax.view_init(elev=30, azim=45)

    # Row 4 -- 4D GradCAM
    ax_4d = fig.add_subplot(gs[3, :])
    all_positions, all_intensities = [], []
    all_classes, all_segments      = [], []

    for i in range(len(sample_idx)):
        cam = cam_maps[i]
        for pos, val in enumerate(cam):
            seg_id = 0
            for s, (lo, hi) in enumerate(zip(
                    segment_bounds[:-1], segment_bounds[1:])):
                if lo <= pos < hi:
                    seg_id = s
                    break
            all_positions.append(pos)
            all_intensities.append(val)
            all_classes.append(y_sample[i])
            all_segments.append(seg_id)

    all_positions   = np.array(all_positions)
    all_intensities = np.array(all_intensities)
    all_classes     = np.array(all_classes)
    all_segments    = np.array(all_segments)

    for seg_id, (seg_name, color) in enumerate(
            zip(segment_names, segment_colors)):
        mask  = all_segments == seg_id
        sizes = np.where(all_classes[mask] == 1, 15, 8)
        ax_4d.scatter(
            all_positions[mask], all_intensities[mask],
            s=sizes, c=color, alpha=0.4,
            label=f"{seg_name} (large=Blocker, small=Non-Blocker)"
        )

    for b in segment_bounds[1:-1]:
        ax_4d.axvline(b, color="red", linestyle="--",
                      linewidth=1.0, alpha=0.6)

    ax_4d.set_title(
        "4D GradCAM -- Position (x) x Activation (y) x "
        "Class (marker size) x Fingerprint Segment (grayscale)",
        fontsize=10, fontweight="bold"
    )
    ax_4d.set_xlabel("Fingerprint Feature Position", fontsize=9)
    ax_4d.set_ylabel("GradCAM Activation (normalised)", fontsize=9)
    ax_4d.legend(fontsize=7, loc="upper right", ncol=3)
    ax_4d.grid(True, alpha=0.3)
    seg_midpoints = [
        (segment_bounds[i] + segment_bounds[i+1]) // 2
        for i in range(len(segment_names))
    ]
    ax_4d.set_xticks(seg_midpoints)
    ax_4d.set_xticklabels(segment_names, fontsize=8)

    plt.savefig(f"{version_name}_gradcam_saliency.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {version_name}_gradcam_saliency.png")

# =============================================================
# VISUALISATION BLOCK 13 -- FILTER VISUALISATION
# =============================================================
print("\n" + "=" * 60)
print("  CONV FILTER VISUALISATION")
print("=" * 60)

for version_name, model in best_models.items():
    conv1, conv2, conv3 = model.get_conv_filters()

    fig, axes = plt.subplots(3, 8, figsize=(20, 8))
    fig.suptitle(f"Learned Conv Filter Patterns -- {version_name}",
                 fontsize=12, fontweight="bold")

    for layer_idx, (conv, n_show, title) in enumerate([
        (conv1, 8, "Conv Layer 1 (64 filters, kernel=7)"),
        (conv2, 8, "Conv Layer 2 (128 filters, kernel=5)"),
        (conv3, 8, "Conv Layer 3 (256 filters, kernel=3)"),
    ]):
        weights = conv.weight.data.cpu().numpy()  # (out, in, kernel)
        for j in range(n_show):
            ax = axes[layer_idx, j]
            filt = weights[j, 0, :]
            ax.bar(np.arange(len(filt)), filt,
                   color=["#424242" if v >= 0 else "#9E9E9E" for v in filt],
                   edgecolor="black", linewidth=0.3)
            ax.axhline(0, color="black", linewidth=0.5)
            ax.set_title(f"F{j+1}", fontsize=7)
            ax.tick_params(labelsize=5)
            if j == 0:
                ax.set_ylabel(f"Layer {layer_idx+1}", fontsize=8)

    plt.tight_layout()
    plt.savefig(f"{version_name}_filter_viz.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {version_name}_filter_viz.png")

# =============================================================
# VISUALISATION BLOCK 14 -- SUMMARY TABLE
# =============================================================
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis("off")
metrics_order = ["ROC-AUC", "PR-AUC", "Accuracy", "F1",
                 "Precision", "Recall", "Brier", "RMSE", "MAE"]
table_data    = []
for name in versions:
    row = [name.replace("CNN_", "")] + \
          [f"{all_results[name][m]:.4f}" for m in metrics_order]
    table_data.append(row)
col_labels = ["Version"] + metrics_order
table      = ax.table(
    cellText  = table_data,
    colLabels = col_labels,
    loc       = "center",
    cellLoc   = "center"
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 2.0)
for j in range(len(col_labels)):
    table[0, j].set_facecolor("#1a237e")
    table[0, j].set_text_props(color="white", fontweight="bold")
row_colors = ["#E3F2FD", "#FBE9E7", "#E8F5E9"]
for i, color in enumerate(row_colors):
    for j in range(len(col_labels)):
        table[i+1, j].set_facecolor(color)
plt.title("CNN Module -- Complete Results Summary",
          fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("CNN_summary_table.png", dpi=150, bbox_inches="tight")
plt.show()

# ── FINAL PRINT ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("  CNN MODULE -- FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"{'Version':<22} {'ROC-AUC':>8} {'PR-AUC':>8} "
      f"{'Acc':>7} {'F1':>7} {'Brier':>8} {'RMSE':>7} {'MAE':>7}")
print("-" * 78)
for name, res in all_results.items():
    target = "  <-- TARGET MET" if res["ROC-AUC"] >= 0.87 else ""
    print(f"{name:<22} {res['ROC-AUC']:>8.4f} {res['PR-AUC']:>8.4f} "
          f"{res['Accuracy']:>7.4f} {res['F1']:>7.4f} "
          f"{res['Brier']:>8.4f} {res['RMSE']:>7.4f} "
          f"{res['MAE']:>7.4f}{target}")

print("\n" + "=" * 60)
print("  STEP 11 COMPLETE -- ALL OUTPUTS SAVED")
print("=" * 60)
saved_files = [
    "CNN_loss_accuracy.png",
    "CNN_auc_f1.png",
    "CNN_precision_recall_curves.png",
    "CNN_calibration_curves.png",
    "CNN_test_curves.png",
    "CNN_all_metrics.png",
    "CNN_confusion_matrices.png",
    "CNN_calibration_reliability.png",
    "CNN_threshold_analysis.png",
    "CNN_bootstrap_ci.png",
    "CNN_tsne.png",
    "CNN_probability_dist.png",
    "CNN_error_analysis.png",
    "CNN_applicability_domain.png",
    "CNN_y_randomisation.png",
    "CNN_Attention_gradcam_saliency.png",
    "CNN_NoAttention_gradcam_saliency.png",
    "CNN_Fingerprint_gradcam_saliency.png",
    "CNN_Attention_filter_viz.png",
    "CNN_NoAttention_filter_viz.png",
    "CNN_Fingerprint_filter_viz.png",
    "CNN_summary_table.png",
]
for f in saved_files:
    print(f"  {f}")

  CNN MODULE -- COMPLETE WITH ALL FEATURES
Device                  : cpu
Input dimension         : 6856
Class weights           : Non-Blocker=2.6831  Blocker=0.6145

  TRAINING : CNN_Attention

  Config: lr=0.001  wd=0.0001  epochs=50
  Epoch 010 | Train Loss: 0.1778  AUC: 0.9745  Acc: 0.9384 | Val Loss: 0.9813  AUC: 0.8102  Acc: 0.7686
  Epoch 020 | Train Loss: 0.1079  AUC: 0.9884  Acc: 0.9676 | Val Loss: 1.6227  AUC: 0.6838  Acc: 0.8395
  Epoch 030 | Train Loss: 0.0492  AUC: 0.9949  Acc: 0.9865 | Val Loss: 1.6245  AUC: 0.6572  Acc: 0.8279
  Epoch 040 | Train Loss: 0.0407  AUC: 0.9964  Acc: 0.9903 | Val Loss: 1.6087  AUC: 0.6574  Acc: 0.8244
  Epoch 050 | Train Loss: 0.0350  AUC: 0.9979  Acc: 0.9918 | Val Loss: 1.6566  AUC: 0.6559  Acc: 0.8256
  Val ROC-AUC : 0.8275

  Config: lr=0.0005  wd=0.0001  epochs=50
  Epoch 010 | Train Loss: 0.1417  AUC: 0.9832  Acc: 0.9509 | Val Loss: 1.0697  AUC: 0.8214  Acc: 0.8140
  Epoch 020 | Train Loss: 0.0411  AUC: 0.9967  Acc: 0.9885 | Val Loss: 1.68

In [16]:
# STEP 12 · GNN MODULE -- COMPLETE WITH ALL FEATURES
# Version 1 -- GINEConv with Attention
# Version 2 -- GINEConv without Attention
# Version 3 -- GINEConv with Fingerprint Features
# Includes  -- GNN GradCAM, Molecular Highlighting,
#              t-SNE, Calibration, Confusion Matrix, Bootstrap CI,
#              Y-Randomisation, Applicability Domain, Threshold Analysis,
#              Error Analysis, Scaffold Analysis

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from torch_geometric.loader import DataLoader as GeoDataLoader
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    f1_score, precision_score, recall_score,
    brier_score_loss, mean_squared_error, mean_absolute_error,
    roc_curve, precision_recall_curve, confusion_matrix,
)
from sklearn.calibration import calibration_curve
from sklearn.manifold import TSNE
from sklearn.utils import resample
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.Scaffolds import MurckoScaffold
import warnings
warnings.filterwarnings("ignore")

print("=" * 60)
print("  GNN MODULE -- COMPLETE WITH ALL FEATURES")
print("=" * 60)

# ── CONSTANTS ─────────────────────────────────────────────────
NODE_DIM      = 55
EDGE_DIM      = 13
HIDDEN_DIM    = 256
LATENT_DIM    = 256
NUM_CLASSES   = 2
FP_DIM        = X_train.shape[1]   # 6856
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_WEIGHTS = torch.tensor([2.6831, 0.6145], dtype=torch.float).to(DEVICE)

print(f"Device                  : {DEVICE}")
print(f"Node feature dim        : {NODE_DIM}")
print(f"Edge feature dim        : {EDGE_DIM}")
print(f"Fingerprint dim         : {FP_DIM}")

# ── ATTACH FINGERPRINTS TO GRAPH OBJECTS ──────────────────────
print("\nAttaching fingerprints to graph objects...")

all_fp = np.vstack([X_train, X_val, X_test])

for i, g in enumerate(graph_data):
    g.fp = torch.tensor(all_fp[i], dtype=torch.float32).unsqueeze(0)

train_graphs_fp = [graph_data[i] for i in idx_train]
val_graphs_fp   = [graph_data[i] for i in idx_val]
test_graphs_fp  = [graph_data[i] for i in idx_test]

train_loader_fp = GeoDataLoader(train_graphs_fp, batch_size=64,
                                shuffle=True,  drop_last=True)
val_loader_fp   = GeoDataLoader(val_graphs_fp,   batch_size=64,
                                shuffle=False, drop_last=False)
test_loader_fp  = GeoDataLoader(test_graphs_fp,  batch_size=64,
                                shuffle=False, drop_last=False)

print("Fingerprints attached to all graph objects")

# ── GRAPH ATTENTION POOLING ───────────────────────────────────
class GraphAttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, batch):
        attn_weights = self.attention(x)
        attn_weights = torch.softmax(attn_weights, dim=0)
        x_weighted   = x * attn_weights
        return global_add_pool(x_weighted, batch)

# ── VERSION 1 -- GINEConv WITH ATTENTION ──────────────────────
class GNN_Attention(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, latent_dim, num_classes):
        super().__init__()
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        def make_mlp(in_d, out_d):
            return nn.Sequential(
                nn.Linear(in_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.ReLU(),
                nn.Linear(out_d, out_d),
            )

        self.conv1 = GINEConv(make_mlp(node_dim,   hidden_dim),
                              edge_dim=hidden_dim)
        self.conv2 = GINEConv(make_mlp(hidden_dim, hidden_dim),
                              edge_dim=hidden_dim)
        self.conv3 = GINEConv(make_mlp(hidden_dim, hidden_dim),
                              edge_dim=hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)
        self.bn3   = nn.BatchNorm1d(hidden_dim)
        self.pool  = GraphAttentionPooling(hidden_dim)
        self.latent_proj = nn.Sequential(
            nn.Linear(hidden_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, data):
        x, ei, ea, batch = (data.x, data.edge_index,
                             data.edge_attr, data.batch)
        eh = self.edge_proj(ea)
        x  = self.bn1(F.relu(self.conv1(x, ei, eh)))
        x  = self.bn2(F.relu(self.conv2(x, ei, eh)))
        x  = self.bn3(F.relu(self.conv3(x, ei, eh)))
        x  = self.pool(x, batch)
        z  = self.latent_proj(x)
        return self.classifier(z), z

    def forward_node(self, data):
        x, ei, ea, batch = (data.x, data.edge_index,
                             data.edge_attr, data.batch)
        eh   = self.edge_proj(ea)
        x    = self.bn1(F.relu(self.conv1(x, ei, eh)))
        x    = self.bn2(F.relu(self.conv2(x, ei, eh)))
        node = self.bn3(F.relu(self.conv3(x, ei, eh)))
        g    = self.pool(node, batch)
        z    = self.latent_proj(g)
        return self.classifier(z), z, node

# ── VERSION 2 -- GINEConv WITHOUT ATTENTION ───────────────────
class GNN_NoAttention(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, latent_dim, num_classes):
        super().__init__()
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        def make_mlp(in_d, out_d):
            return nn.Sequential(
                nn.Linear(in_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.ReLU(),
                nn.Linear(out_d, out_d),
            )

        self.conv1 = GINEConv(make_mlp(node_dim,   hidden_dim),
                              edge_dim=hidden_dim)
        self.conv2 = GINEConv(make_mlp(hidden_dim, hidden_dim),
                              edge_dim=hidden_dim)
        self.conv3 = GINEConv(make_mlp(hidden_dim, hidden_dim),
                              edge_dim=hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)
        self.bn3   = nn.BatchNorm1d(hidden_dim)
        self.latent_proj = nn.Sequential(
            nn.Linear(hidden_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, data):
        x, ei, ea, batch = (data.x, data.edge_index,
                             data.edge_attr, data.batch)
        eh = self.edge_proj(ea)
        x  = self.bn1(F.relu(self.conv1(x, ei, eh)))
        x  = self.bn2(F.relu(self.conv2(x, ei, eh)))
        x  = self.bn3(F.relu(self.conv3(x, ei, eh)))
        x  = global_mean_pool(x, batch)
        z  = self.latent_proj(x)
        return self.classifier(z), z

    def forward_node(self, data):
        x, ei, ea, batch = (data.x, data.edge_index,
                             data.edge_attr, data.batch)
        eh   = self.edge_proj(ea)
        x    = self.bn1(F.relu(self.conv1(x, ei, eh)))
        x    = self.bn2(F.relu(self.conv2(x, ei, eh)))
        node = self.bn3(F.relu(self.conv3(x, ei, eh)))
        g    = global_mean_pool(node, batch)
        z    = self.latent_proj(g)
        return self.classifier(z), z, node

# ── VERSION 3 -- GINEConv WITH FINGERPRINT FUSION ─────────────
class GNN_Fingerprint(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, latent_dim,
                 num_classes, fp_dim=6856):
        super().__init__()
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        def make_mlp(in_d, out_d):
            return nn.Sequential(
                nn.Linear(in_d, out_d),
                nn.BatchNorm1d(out_d),
                nn.ReLU(),
                nn.Linear(out_d, out_d),
            )

        self.conv1 = GINEConv(make_mlp(node_dim,   hidden_dim),
                              edge_dim=hidden_dim)
        self.conv2 = GINEConv(make_mlp(hidden_dim, hidden_dim),
                              edge_dim=hidden_dim)
        self.conv3 = GINEConv(make_mlp(hidden_dim, hidden_dim),
                              edge_dim=hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)
        self.bn3   = nn.BatchNorm1d(hidden_dim)
        self.pool  = GraphAttentionPooling(hidden_dim)

        # Fingerprint branch
        self.fp_branch = nn.Sequential(
            nn.Linear(fp_dim,    hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Fusion -- graph embedding + fingerprint embedding
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim + latent_dim, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, data):
        x, ei, ea, batch = (data.x, data.edge_index,
                             data.edge_attr, data.batch)
        eh      = self.edge_proj(ea)
        x       = self.bn1(F.relu(self.conv1(x, ei, eh)))
        x       = self.bn2(F.relu(self.conv2(x, ei, eh)))
        x       = self.bn3(F.relu(self.conv3(x, ei, eh)))
        g       = self.pool(x, batch)
        fp      = data.fp.to(DEVICE).squeeze(1)
        fp_feat = self.fp_branch(fp)
        z       = self.fusion(torch.cat([g, fp_feat], dim=1))
        return self.classifier(z), z

    def forward_node(self, data):
        x, ei, ea, batch = (data.x, data.edge_index,
                             data.edge_attr, data.batch)
        eh      = self.edge_proj(ea)
        x       = self.bn1(F.relu(self.conv1(x, ei, eh)))
        x       = self.bn2(F.relu(self.conv2(x, ei, eh)))
        node    = self.bn3(F.relu(self.conv3(x, ei, eh)))
        g       = self.pool(node, batch)
        fp      = data.fp.to(DEVICE).squeeze(1)
        fp_feat = self.fp_branch(fp)
        z       = self.fusion(torch.cat([g, fp_feat], dim=1))
        return self.classifier(z), z, node

# ── GNN GRADCAM ───────────────────────────────────────────────
class GNNGradCAM:
    def __init__(self, model):
        self.model = model

    def compute(self, data, target_class):
        self.model.eval()
        data   = data.to(DEVICE)
        data.x = data.x.detach().requires_grad_(True)
        logits, _, node_out = self.model.forward_node(data)
        self.model.zero_grad()
        logits[0, target_class].backward()
        grads = data.x.grad
        if grads is None:
            return np.zeros(data.x.shape[0])
        importance = grads.abs().mean(dim=1).detach().cpu().numpy()
        if importance.max() > 0:
            importance = importance / importance.max()
        return importance

# ── TRAINING FUNCTION -- ALL METRICS ──────────────────────────
def train_gnn(model, train_loader, val_loader, epochs, lr, weight_decay):
    model     = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode="max", patience=5, factor=0.5)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

    history  = {
        "train_loss" : [], "val_loss"  : [],
        "train_auc"  : [], "val_auc"   : [],
        "train_acc"  : [], "val_acc"   : [],
        "train_f1"   : [], "val_f1"    : [],
        "train_prec" : [], "val_prec"  : [],
        "train_rec"  : [], "val_rec"   : [],
        "train_brier": [], "val_brier" : [],
        "train_rmse" : [], "val_rmse"  : [],
        "train_mae"  : [], "val_mae"   : [],
    }

    best_val_auc = 0.0
    best_state   = None

    for epoch in range(epochs):
        model.train()
        t_losses, t_probs, t_preds, t_labels = [], [], [], []
        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(batch)
            loss      = criterion(logits, batch.y.squeeze())
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            t_losses.append(loss.item())
            t_probs.extend(
                F.softmax(logits, dim=1)[:,1].detach().cpu().numpy())
            t_preds.extend(
                logits.argmax(dim=1).detach().cpu().numpy())
            t_labels.extend(batch.y.squeeze().cpu().numpy())

        model.eval()
        v_losses, v_probs, v_preds, v_labels = [], [], [], []
        with torch.no_grad():
            for batch in val_loader:
                batch     = batch.to(DEVICE)
                logits, _ = model(batch)
                loss      = criterion(logits, batch.y.squeeze())
                v_losses.append(loss.item())
                v_probs.extend(
                    F.softmax(logits, dim=1)[:,1].cpu().numpy())
                v_preds.extend(
                    logits.argmax(dim=1).cpu().numpy())
                v_labels.extend(batch.y.squeeze().cpu().numpy())

        tp, tl, td = np.array(t_probs), np.array(t_labels), np.array(t_preds)
        vp, vl, vd = np.array(v_probs), np.array(v_labels), np.array(v_preds)

        history["train_loss"].append(np.mean(t_losses))
        history["val_loss"].append(np.mean(v_losses))
        history["train_auc"].append(roc_auc_score(tl, tp))
        history["val_auc"].append(roc_auc_score(vl, vp))
        history["train_acc"].append(accuracy_score(tl, td))
        history["val_acc"].append(accuracy_score(vl, vd))
        history["train_f1"].append(f1_score(tl, td, zero_division=0))
        history["val_f1"].append(f1_score(vl, vd, zero_division=0))
        history["train_prec"].append(precision_score(tl, td, zero_division=0))
        history["val_prec"].append(precision_score(vl, vd, zero_division=0))
        history["train_rec"].append(recall_score(tl, td, zero_division=0))
        history["val_rec"].append(recall_score(vl, vd, zero_division=0))
        history["train_brier"].append(brier_score_loss(tl, tp))
        history["val_brier"].append(brier_score_loss(vl, vp))
        history["train_rmse"].append(np.sqrt(mean_squared_error(tl, tp)))
        history["val_rmse"].append(np.sqrt(mean_squared_error(vl, vp)))
        history["train_mae"].append(mean_absolute_error(tl, tp))
        history["val_mae"].append(mean_absolute_error(vl, vp))

        scheduler.step(history["val_auc"][-1])

        if history["val_auc"][-1] > best_val_auc:
            best_val_auc = history["val_auc"][-1]
            best_state   = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:03d} | "
                  f"Train Loss: {history['train_loss'][-1]:.4f}  "
                  f"AUC: {history['train_auc'][-1]:.4f}  "
                  f"Acc: {history['train_acc'][-1]:.4f} | "
                  f"Val Loss: {history['val_loss'][-1]:.4f}  "
                  f"AUC: {history['val_auc'][-1]:.4f}  "
                  f"Acc: {history['val_acc'][-1]:.4f}")

    model.load_state_dict(best_state)
    return model, history

# ── GNN EVALUATION FUNCTION ───────────────────────────────────
def evaluate_gnn(model, loader):
    model.eval()
    all_probs, all_preds, all_labels, all_latents = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            batch     = batch.to(DEVICE)
            logits, z = model(batch)
            probs     = F.softmax(logits, dim=1)[:,1].cpu().numpy()
            preds     = logits.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(batch.y.squeeze().cpu().numpy())
            all_latents.append(z.cpu().numpy())

    p    = np.array(all_probs)
    y    = np.array(all_labels)
    pred = np.array(all_preds)
    Z    = np.concatenate(all_latents, axis=0)

    return {
        "ROC-AUC"   : roc_auc_score(y, p),
        "PR-AUC"    : average_precision_score(y, p),
        "Accuracy"  : accuracy_score(y, pred),
        "F1"        : f1_score(y, pred),
        "Precision" : precision_score(y, pred),
        "Recall"    : recall_score(y, pred),
        "Brier"     : brier_score_loss(y, p),
        "RMSE"      : np.sqrt(mean_squared_error(y, p)),
        "MAE"       : mean_absolute_error(y, p),
        "probs"     : p,
        "labels"    : y,
        "preds"     : pred,
        "latents"   : Z,
    }

# ── BOOTSTRAP CI ──────────────────────────────────────────────
def bootstrap_ci(y_true, y_prob, metric_fn, n_boot=1000, ci=95):
    scores = []
    for _ in range(n_boot):
        idx = resample(np.arange(len(y_true)), random_state=None)
        try:
            scores.append(metric_fn(y_true[idx], y_prob[idx]))
        except Exception:
            pass
    lower = np.percentile(scores, (100 - ci) / 2)
    upper = np.percentile(scores, 100 - (100 - ci) / 2)
    return np.mean(scores), lower, upper

# ── Y-RANDOMISATION ───────────────────────────────────────────
def y_randomisation_gnn(model_class, train_graphs, val_loader,
                         best_config, n_runs=5, use_fp=False):
    rand_aucs = []
    print("  Running Y-Randomisation test...")
    for run in range(n_runs):
        if use_fp:
            rand_model = model_class(NODE_DIM, EDGE_DIM, HIDDEN_DIM,
                                     LATENT_DIM, NUM_CLASSES, fp_dim=FP_DIM)
        else:
            rand_model = model_class(NODE_DIM, EDGE_DIM, HIDDEN_DIM,
                                     LATENT_DIM, NUM_CLASSES)
        rand_graphs = []
        for g in train_graphs:
            g_copy   = g.clone()
            g_copy.y = torch.tensor(
                [np.random.randint(0, 2)], dtype=torch.long)
            rand_graphs.append(g_copy)
        rand_loader = GeoDataLoader(rand_graphs,
                                    batch_size=64, shuffle=True)
        rand_model, _ = train_gnn(
            rand_model, rand_loader, val_loader,
            epochs       = 20,
            lr           = best_config["lr"],
            weight_decay = best_config["weight_decay"]
        )
        metrics = evaluate_gnn(rand_model, val_loader)
        rand_aucs.append(metrics["ROC-AUC"])
        print(f"    Run {run+1}: Val ROC-AUC = {metrics['ROC-AUC']:.4f}")
    return np.array(rand_aucs)

# ── APPLICABILITY DOMAIN ──────────────────────────────────────
def applicability_domain_gnn(train_graphs, test_graphs, threshold_pct=95):
    from sklearn.metrics.pairwise import cosine_similarity

    def graph_fp(g):
        return g.x.mean(dim=0).numpy()

    train_fps = np.stack([graph_fp(g) for g in train_graphs])
    test_fps  = np.stack([graph_fp(g) for g in test_graphs])
    sim       = cosine_similarity(test_fps, train_fps)
    mean_sim  = sim.max(axis=1)
    threshold = np.percentile(
        cosine_similarity(train_fps, train_fps).mean(axis=1),
        100 - threshold_pct
    )
    return mean_sim, threshold, mean_sim >= threshold

# ── MANUAL GRID SEARCH ────────────────────────────────────────
GRID = [
    {"lr": 1e-3, "weight_decay": 1e-4, "epochs": 50},
    {"lr": 5e-4, "weight_decay": 1e-4, "epochs": 50},
    {"lr": 1e-4, "weight_decay": 1e-5, "epochs": 50},
]

# Versions 1 and 2 use standard graph loaders
# Version 3 uses fingerprint-attached graph loaders
GNN_VERSIONS = {
    "GNN_Attention"   : (GNN_Attention,   train_loader,    val_loader,
                          test_loader,   False),
    "GNN_NoAttention" : (GNN_NoAttention, train_loader,    val_loader,
                          test_loader,   False),
    "GNN_Fingerprint" : (GNN_Fingerprint, train_loader_fp, val_loader_fp,
                          test_loader_fp, True),
}

gnn_results   = {}
gnn_histories = {}
gnn_models    = {}
gnn_configs   = {}

for version_name, (ModelClass, t_loader,
                   v_loader, te_loader, use_fp) in GNN_VERSIONS.items():
    print("\n" + "=" * 60)
    print(f"  TRAINING : {version_name}")
    print("=" * 60)

    best_auc     = 0.0
    best_config  = None
    best_model_v = None
    best_history = None

    for config in GRID:
        print(f"\n  Config: lr={config['lr']}  "
              f"wd={config['weight_decay']}  "
              f"epochs={config['epochs']}")
        if use_fp:
            model = ModelClass(NODE_DIM, EDGE_DIM, HIDDEN_DIM,
                               LATENT_DIM, NUM_CLASSES, fp_dim=FP_DIM)
        else:
            model = ModelClass(NODE_DIM, EDGE_DIM, HIDDEN_DIM,
                               LATENT_DIM, NUM_CLASSES)
        model, history = train_gnn(
            model, t_loader, v_loader,
            epochs       = config["epochs"],
            lr           = config["lr"],
            weight_decay = config["weight_decay"]
        )
        val_metrics = evaluate_gnn(model, v_loader)
        print(f"  Val ROC-AUC : {val_metrics['ROC-AUC']:.4f}")

        if val_metrics["ROC-AUC"] > best_auc:
            best_auc     = val_metrics["ROC-AUC"]
            best_config  = config
            best_model_v = model
            best_history = history

    gnn_models[version_name]    = best_model_v
    gnn_histories[version_name] = best_history
    gnn_configs[version_name]   = best_config
    test_metrics                = evaluate_gnn(best_model_v, te_loader)
    gnn_results[version_name]   = test_metrics

    print(f"\n  Best config    : {best_config}")
    print(f"  Test ROC-AUC   : {test_metrics['ROC-AUC']:.4f}")
    print(f"  Test PR-AUC    : {test_metrics['PR-AUC']:.4f}")
    print(f"  Test Accuracy  : {test_metrics['Accuracy']:.4f}")
    print(f"  Test F1        : {test_metrics['F1']:.4f}")
    print(f"  Test Precision : {test_metrics['Precision']:.4f}")
    print(f"  Test Recall    : {test_metrics['Recall']:.4f}")
    print(f"  Test Brier     : {test_metrics['Brier']:.4f}")
    print(f"  Test RMSE      : {test_metrics['RMSE']:.4f}")
    print(f"  Test MAE       : {test_metrics['MAE']:.4f}")

# ── COLOURS ───────────────────────────────────────────────────
COLORS_GNN = {
    "GNN_Attention"   : "#9C27B0",
    "GNN_NoAttention" : "#FF9800",
    "GNN_Fingerprint" : "#009688",
}
versions_gnn = list(gnn_histories.keys())

# =============================================================
# VISUALISATION BLOCK 1 -- TRAINING CURVES
# =============================================================
for pairs, title, fname in [
    ([("train_loss", "val_loss", "Loss",      "Loss"),
      ("train_acc",  "val_acc",  "Accuracy",  "Accuracy")],
     "Loss & Accuracy", "GNN_loss_accuracy.png"),
    ([("train_auc",  "val_auc",  "ROC-AUC",   "ROC-AUC"),
      ("train_f1",   "val_f1",   "F1",        "F1 Score")],
     "ROC-AUC & F1", "GNN_auc_f1.png"),
    ([("train_prec", "val_prec", "Precision", "Precision"),
      ("train_rec",  "val_rec",  "Recall",    "Recall")],
     "Precision & Recall", "GNN_precision_recall_curves.png"),
]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (t_key, v_key, label, ylabel) in zip(axes, pairs):
        for name in versions_gnn:
            ax.plot(gnn_histories[name][t_key], color=COLORS_GNN[name],
                    linestyle="--", alpha=0.5, linewidth=1.2,
                    label=f"{name} Train")
            ax.plot(gnn_histories[name][v_key], color=COLORS_GNN[name],
                    linestyle="-", linewidth=2, label=f"{name} Val")
        if "loss" not in t_key:
            ax.axhline(0.87, color="red", linestyle="--",
                       linewidth=1.5, label="Target (0.87)")
        ax.set_title(f"{label} Curves", fontsize=12, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    plt.suptitle(f"GNN -- {title}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()

# Calibration metric curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (t_key, v_key, title) in zip(axes, [
    ("train_brier", "val_brier", "Brier Score"),
    ("train_rmse",  "val_rmse",  "RMSE"),
    ("train_mae",   "val_mae",   "MAE"),
]):
    for name in versions_gnn:
        ax.plot(gnn_histories[name][t_key], color=COLORS_GNN[name],
                linestyle="--", alpha=0.5, linewidth=1.2,
                label=f"{name} Train")
        ax.plot(gnn_histories[name][v_key], color=COLORS_GNN[name],
                linestyle="-", linewidth=2, label=f"{name} Val")
    ax.set_title(f"{title} (lower is better)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle("GNN -- Calibration Metric Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_calibration_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 2 -- TEST SET CURVES
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for name, res in gnn_results.items():
    fpr, tpr, _ = roc_curve(res["labels"], res["probs"])
    optimal_idx = np.argmax(tpr - fpr)
    ax.plot(fpr, tpr, color=COLORS_GNN[name], linewidth=2.5,
            label=f"{name} (AUC={res['ROC-AUC']:.4f})")
    ax.scatter(fpr[optimal_idx], tpr[optimal_idx],
               color=COLORS_GNN[name], s=100, zorder=5, marker="*")
    ax.fill_between(fpr, tpr, alpha=0.05, color=COLORS_GNN[name])
ax.plot([0,1],[0,1], "k--", alpha=0.4)
ax.set_title("ROC Curves -- Test Set", fontsize=12, fontweight="bold")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[1]
for name, res in gnn_results.items():
    prec, rec, _ = precision_recall_curve(res["labels"], res["probs"])
    ax.plot(rec, prec, color=COLORS_GNN[name], linewidth=2.5,
            label=f"{name} (PR-AUC={res['PR-AUC']:.4f})")
    ax.fill_between(rec, prec, alpha=0.05, color=COLORS_GNN[name])
baseline = list(gnn_results.values())[0]["labels"].mean()
ax.axhline(baseline, color="red", linestyle="--",
           linewidth=1.5, label=f"Baseline ({baseline:.2f})")
ax.set_title("PR Curves -- Test Set", fontsize=12, fontweight="bold")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
plt.suptitle("GNN -- Test Set Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_test_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 3 -- ALL 9 METRICS
# =============================================================
all_metric_cfg = {
    "ROC-AUC"   : {"higher": True,  "target": 0.87},
    "PR-AUC"    : {"higher": True,  "target": 0.87},
    "Accuracy"  : {"higher": True,  "target": 0.87},
    "F1"        : {"higher": True,  "target": 0.87},
    "Precision" : {"higher": True,  "target": 0.87},
    "Recall"    : {"higher": True,  "target": 0.87},
    "Brier"     : {"higher": False, "target": 0.15},
    "RMSE"      : {"higher": False, "target": 0.35},
    "MAE"       : {"higher": False, "target": 0.25},
}
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes_flat = axes.flatten()
x, width  = np.arange(len(versions_gnn)), 0.5
for i, (metric, info) in enumerate(all_metric_cfg.items()):
    ax   = axes_flat[i]
    vals = [gnn_results[v][metric] for v in versions_gnn]
    bars = ax.bar(x, vals, width,
                  color=[COLORS_GNN[v] for v in versions_gnn],
                  edgecolor="black", alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{val:.4f}", ha="center",
                va="bottom", fontsize=9, fontweight="bold")
    direction = "higher" if info["higher"] else "lower"
    ax.axhline(info["target"], color="red", linestyle="--",
               linewidth=1.5,
               label=f"Target ({info['target']}) -- {direction} is better")
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(
        [v.replace("GNN_", "") for v in versions_gnn], fontsize=9)
    ax.set_ylabel(metric)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_ylim(0, max(vals) * 1.25 if not info["higher"] else 1.12)
plt.suptitle("GNN -- All 9 Metrics (Test Set)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_all_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 4 -- CONFUSION MATRICES
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, gnn_results.items()):
    cm      = confusion_matrix(res["labels"], res["preds"])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im      = ax.imshow(cm_norm, cmap="gray_r", vmin=0, vmax=1)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Non-Blocker", "Blocker"], fontsize=9)
    ax.set_yticklabels(["Non-Blocker", "Blocker"], fontsize=9)
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("Actual",    fontsize=10)
    ax.set_title(f"Confusion Matrix\n{name}", fontsize=11, fontweight="bold")
    for r in range(2):
        for c in range(2):
            ax.text(c, r,
                    f"{cm[r,c]}\n({cm_norm[r,c]:.2f})",
                    ha="center", va="center",
                    fontsize=11, fontweight="bold",
                    color="white" if cm_norm[r,c] > 0.5 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("GNN -- Confusion Matrices (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 5 -- CALIBRATION CURVES
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, gnn_results.items()):
    frac_pos, mean_pred = calibration_curve(
        res["labels"], res["probs"], n_bins=10)
    ax.plot(mean_pred, frac_pos, "s-", color=COLORS_GNN[name],
            linewidth=2, markersize=6, label=name)
    ax.plot([0,1],[0,1], "k--", alpha=0.5, label="Perfect calibration")
    ax.fill_between(mean_pred, frac_pos, mean_pred,
                    alpha=0.1, color=COLORS_GNN[name])
    ax.set_title(f"Calibration Curve\n{name}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Fraction of Positives")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
plt.suptitle("GNN -- Reliability Diagrams (Calibration)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_calibration_reliability.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 6 -- THRESHOLD ANALYSIS
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, gnn_results.items()):
    thresholds = np.linspace(0.1, 0.9, 50)
    accs, f1s, precs, recs = [], [], [], []
    for thr in thresholds:
        pred_thr = (res["probs"] >= thr).astype(int)
        accs.append(accuracy_score(res["labels"], pred_thr))
        f1s.append(f1_score(res["labels"], pred_thr, zero_division=0))
        precs.append(precision_score(res["labels"], pred_thr, zero_division=0))
        recs.append(recall_score(res["labels"], pred_thr, zero_division=0))
    ax.plot(thresholds, accs,  color="#2196F3", linewidth=2, label="Accuracy")
    ax.plot(thresholds, f1s,   color="#4CAF50", linewidth=2, label="F1")
    ax.plot(thresholds, precs, color="#FF5722", linewidth=2, label="Precision")
    ax.plot(thresholds, recs,  color="#9C27B0", linewidth=2, label="Recall")
    ax.axvline(0.5, color="black", linestyle="--",
               linewidth=1.5, label="Default (0.5)")
    best_thr = thresholds[np.argmax(f1s)]
    ax.axvline(best_thr, color="green", linestyle=":",
               linewidth=1.5, label=f"Optimal F1 ({best_thr:.2f})")
    ax.set_title(f"Threshold Analysis\n{name}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Decision Threshold")
    ax.set_ylabel("Score")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
plt.suptitle("GNN -- Threshold Analysis (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_threshold_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 7 -- BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================
print("\n" + "=" * 60)
print("  BOOTSTRAP CONFIDENCE INTERVALS (1000 iterations)")
print("=" * 60)

gnn_bootstrap = {}
for name, res in gnn_results.items():
    print(f"\n  {name}")
    ci_dict = {}
    for metric, fn in [
        ("ROC-AUC",  lambda y, p: roc_auc_score(y, p)),
        ("PR-AUC",   lambda y, p: average_precision_score(y, p)),
        ("Accuracy", lambda y, p: accuracy_score(y, (p >= 0.5).astype(int))),
        ("F1",       lambda y, p: f1_score(y, (p >= 0.5).astype(int),
                                           zero_division=0)),
    ]:
        mean, lo, hi = bootstrap_ci(res["labels"], res["probs"], fn)
        ci_dict[metric] = (mean, lo, hi)
        print(f"    {metric:<12} : {mean:.4f}  "
              f"95% CI [{lo:.4f} -- {hi:.4f}]")
    gnn_bootstrap[name] = ci_dict

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
ci_metrics = ["ROC-AUC", "PR-AUC", "Accuracy", "F1"]
for ax, metric in zip(axes, ci_metrics):
    means = [gnn_bootstrap[v][metric][0] for v in versions_gnn]
    lows  = [gnn_bootstrap[v][metric][1] for v in versions_gnn]
    highs = [gnn_bootstrap[v][metric][2] for v in versions_gnn]
    yerr  = [[m - l for m, l in zip(means, lows)],
              [h - m for m, h in zip(means, highs)]]
    ax.bar(np.arange(len(versions_gnn)), means, 0.5,
           color=[COLORS_GNN[v] for v in versions_gnn],
           edgecolor="black", alpha=0.85,
           yerr=yerr, capsize=6, error_kw={"linewidth": 2})
    ax.axhline(0.87, color="red", linestyle="--",
               linewidth=1.5, label="Target (0.87)")
    ax.set_title(f"{metric}\n(95% Bootstrap CI)",
                 fontsize=11, fontweight="bold")
    ax.set_xticks(np.arange(len(versions_gnn)))
    ax.set_xticklabels([v.replace("GNN_", "") for v in versions_gnn],
                       fontsize=9, rotation=15)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.12)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, axis="y")
plt.suptitle("GNN -- Bootstrap Confidence Intervals (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_bootstrap_ci.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 8 -- t-SNE LATENT SPACE
# =============================================================
print("\n" + "=" * 60)
print("  t-SNE LATENT SPACE VISUALISATION")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, gnn_results.items()):
    print(f"  Computing t-SNE for {name}...")
    tsne = TSNE(n_components=2, random_state=42,
                perplexity=30, max_iter=1000)
    Z_2d = tsne.fit_transform(res["latents"])
    for lbl, label_name, color in [
        (0, "Non-Blocker", "#FF5722"),
        (1, "Blocker",     "#9C27B0")
    ]:
        mask = res["labels"] == lbl
        ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                   c=color, s=8, alpha=0.6,
                   label=f"{label_name} (n={mask.sum()})")
    ax.set_title(f"t-SNE Latent Space\n{name}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("t-SNE Dimension 1")
    ax.set_ylabel("t-SNE Dimension 2")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
plt.suptitle("GNN -- t-SNE Latent Space (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 9 -- ERROR ANALYSIS
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, gnn_results.items()):
    correct   = res["labels"] == res["preds"]
    incorrect = ~correct
    ax.scatter(np.where(correct)[0], res["probs"][correct],
               c="#4CAF50", s=8, alpha=0.5,
               label=f"Correct ({correct.sum()})")
    ax.scatter(np.where(incorrect)[0], res["probs"][incorrect],
               c="#F44336", s=15, alpha=0.8, marker="x",
               label=f"Misclassified ({incorrect.sum()})")
    ax.axhline(0.5, color="black", linestyle="--",
               linewidth=1.0, alpha=0.5)
    ax.set_title(f"Error Analysis\n{name}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Sample Index")
    ax.set_ylabel("Predicted Probability (Blocker)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle("GNN -- Error Analysis (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 10 -- APPLICABILITY DOMAIN
# =============================================================
print("\n" + "=" * 60)
print("  APPLICABILITY DOMAIN ANALYSIS")
print("=" * 60)

sim_scores, ad_threshold, in_domain = applicability_domain_gnn(
    train_graphs, test_graphs
)
print(f"  AD threshold            : {ad_threshold:.4f}")
print(f"  Test molecules in domain: {in_domain.sum()} / {len(in_domain)} "
      f"({in_domain.mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.hist(sim_scores[in_domain], bins=40, color="#4CAF50",
        alpha=0.7, edgecolor="black", label="In domain")
ax.hist(sim_scores[~in_domain], bins=40, color="#F44336",
        alpha=0.7, edgecolor="black", label="Out of domain")
ax.axvline(ad_threshold, color="black", linestyle="--",
           linewidth=2, label=f"Threshold ({ad_threshold:.3f})")
ax.set_title("Applicability Domain\nSimilarity Distribution",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Max Cosine Similarity to Training Set")
ax.set_ylabel("Count")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
best_name = max(gnn_results, key=lambda n: gnn_results[n]["ROC-AUC"])
res       = gnn_results[best_name]
ax.scatter(sim_scores[in_domain], res["probs"][in_domain],
           c="#4CAF50", s=10, alpha=0.5, label="In domain")
ax.scatter(sim_scores[~in_domain], res["probs"][~in_domain],
           c="#F44336", s=10, alpha=0.5, label="Out of domain")
ax.axvline(ad_threshold, color="black", linestyle="--", linewidth=1.5)
ax.set_title(f"AD vs Prediction Confidence\n{best_name}",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Similarity to Training Set")
ax.set_ylabel("Predicted Probability (Blocker)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.suptitle("GNN -- Applicability Domain Analysis",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_applicability_domain.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 11 -- Y-RANDOMISATION
# =============================================================
print("\n" + "=" * 60)
print("  Y-RANDOMISATION TEST")
print("=" * 60)

gnn_rand          = {}
gnn_version_meta  = {
    "GNN_Attention"   : (GNN_Attention,   train_graphs,    val_loader,    False),
    "GNN_NoAttention" : (GNN_NoAttention, train_graphs,    val_loader,    False),
    "GNN_Fingerprint" : (GNN_Fingerprint, train_graphs_fp, val_loader_fp, True),
}

for version_name, (ModelClass, t_graphs,
                   v_loader, use_fp) in gnn_version_meta.items():
    print(f"\n  {version_name}")
    rand_aucs = y_randomisation_gnn(
        ModelClass, t_graphs, v_loader,
        gnn_configs[version_name], n_runs=5, use_fp=use_fp
    )
    gnn_rand[version_name] = rand_aucs
    real_auc = gnn_results[version_name]["ROC-AUC"]
    print(f"  Real ROC-AUC            : {real_auc:.4f}")
    print(f"  Random mean ROC-AUC     : {rand_aucs.mean():.4f}")
    print(f"  Separation              : {real_auc - rand_aucs.mean():.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(versions_gnn))
for i, name in enumerate(versions_gnn):
    real_auc  = gnn_results[name]["ROC-AUC"]
    rand_aucs = gnn_rand[name]
    ax.bar(i - 0.2, real_auc, 0.35,
           color=COLORS_GNN[name], edgecolor="black", alpha=0.9,
           label=f"{name} Real")
    ax.bar(i + 0.2, rand_aucs.mean(), 0.35,
           color=COLORS_GNN[name], edgecolor="black",
           alpha=0.4, hatch="//", label=f"{name} Random")
    ax.errorbar(i + 0.2, rand_aucs.mean(),
                yerr=rand_aucs.std(),
                fmt="none", color="black", capsize=5)
ax.axhline(0.87, color="red", linestyle="--",
           linewidth=1.5, label="Target (0.87)")
ax.axhline(0.5,  color="gray", linestyle=":",
           linewidth=1.0, label="Random chance (0.5)")
ax.set_xticks(x_pos)
ax.set_xticklabels([v.replace("GNN_", "") for v in versions_gnn], fontsize=10)
ax.set_ylabel("ROC-AUC")
ax.set_title("Y-Randomisation Test -- Real vs Random Labels",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("GNN_y_randomisation.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 12 -- GNN GRADCAM 3D/4D
# =============================================================
print("\n" + "=" * 60)
print("  GNN GRADCAM + MOLECULAR HIGHLIGHTING")
print("=" * 60)

blocker_graphs    = [g for g in test_graphs if g.y.item() == 1][:3]
nonblocker_graphs = [g for g in test_graphs if g.y.item() == 0][:3]
graph_samples     = blocker_graphs + nonblocker_graphs

test_smiles_all   = df["SMILES"].values
blocker_smiles    = [s for s, g in zip(test_smiles_all, test_graphs)
                     if g.y.item() == 1][:3]
nonblocker_smiles = [s for s, g in zip(test_smiles_all, test_graphs)
                     if g.y.item() == 0][:3]
sample_smiles     = blocker_smiles + nonblocker_smiles

feat_group_names  = [
    "AtomType", "Hybrid", "Degree", "Charge", "NumH",
    "Ring",     "Chirality", "Electro", "VdW",  "PartChg"
]
feat_group_colors = [
    "#212121", "#424242", "#616161", "#757575", "#9E9E9E",
    "#BDBDBD", "#CCCCCC", "#D8D8D8", "#E8E8E8", "#F5F5F5"
]

for version_name, model in gnn_models.items():
    print(f"\n  GradCAM for {version_name}...")
    gradcam  = GNNGradCAM(model)
    cam_maps = []

    for g in graph_samples:
        cam = gradcam.compute(g, target_class=g.y.item())
        cam_maps.append(cam)

    max_nodes = max(len(c) for c in cam_maps)

    fig = plt.figure(figsize=(24, 22))
    fig.suptitle(
        f"GNN GradCAM -- Node Importance | {version_name}\n"
        f"hERG Cardiotoxicity Prediction",
        fontsize=14, fontweight="bold"
    )
    gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.55, wspace=0.4)

    # Row 1 -- 2D grayscale heatmap
    for i in range(3):
        ax  = fig.add_subplot(gs[0, i])
        cam = cam_maps[i]
        padded  = np.zeros(max_nodes)
        padded[:len(cam)] = cam
        grid_w  = max(1, int(np.sqrt(max_nodes)))
        total   = grid_w * (max_nodes // grid_w + 1)
        padded2 = np.zeros(total)
        padded2[:len(padded)] = padded
        cam_2d  = padded2.reshape(-1, grid_w)
        ax.imshow(cam_2d, cmap="gray", aspect="auto", interpolation="bilinear")
        ax.set_title(
            f"Sample {i+1}\n"
            f"{'Blocker' if graph_samples[i].y.item()==1 else 'Non-Blocker'} | "
            f"{len(cam)} atoms",
            fontsize=9, fontweight="bold"
        )
        ax.set_xlabel("Atom Col", fontsize=7)
        ax.set_ylabel("Atom Row", fontsize=7)
        ax.tick_params(labelsize=6)

    # Row 2 -- Molecular highlighting
    for i in range(3):
        ax  = fig.add_subplot(gs[1, i])
        smi = sample_smiles[i]
        cam = cam_maps[i]
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            atom_colors = {}
            for atom_idx in range(mol.GetNumAtoms()):
                imp  = cam[atom_idx] if atom_idx < len(cam) else 0.0
                gray = 1.0 - (imp * 0.7)
                atom_colors[atom_idx] = (gray, gray, gray)
            highlight_atoms = list(range(min(mol.GetNumAtoms(), len(cam))))
            try:
                drawer = rdMolDraw2D.MolDraw2DSVG(300, 250)
                drawer.drawOptions().addAtomIndices = False
                rdMolDraw2D.PrepareAndDrawMolecule(
                    drawer, mol,
                    highlightAtoms=highlight_atoms,
                    highlightAtomColors=atom_colors,
                )
                drawer.FinishDrawing()
                svg = drawer.GetDrawingText()
                import io
                from cairosvg import svg2png
                png_data = svg2png(bytestring=svg.encode())
                img      = plt.imread(io.BytesIO(png_data))
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5,
                        f"SMILES:\n{smi[:40]}...\nAtoms: {mol.GetNumAtoms()}",
                        ha="center", va="center",
                        transform=ax.transAxes, fontsize=7)
        ax.set_title(
            f"Mol Highlight {i+1}\n"
            f"{'Blocker' if graph_samples[i].y.item()==1 else 'Non-Blocker'}",
            fontsize=9, fontweight="bold"
        )
        ax.axis("off")

    # Row 3 -- 3D surface
    for i in range(3):
        ax  = fig.add_subplot(gs[2, i], projection="3d")
        cam = cam_maps[i]
        padded  = np.zeros(max_nodes)
        padded[:len(cam)] = cam
        grid_w  = max(1, int(np.sqrt(max_nodes)))
        total   = grid_w * (max_nodes // grid_w + 1)
        padded2 = np.zeros(total)
        padded2[:len(padded)] = padded
        cam_2d  = padded2.reshape(-1, grid_w)
        X_g     = np.arange(cam_2d.shape[1])
        Y_g     = np.arange(cam_2d.shape[0])
        Xm, Ym  = np.meshgrid(X_g, Y_g)
        ax.plot_surface(Xm, Ym, cam_2d,
                        cmap="gray", edgecolor="none", alpha=0.9)
        ax.set_title(
            f"3D Surface\n"
            f"{'Blocker' if graph_samples[i].y.item()==1 else 'Non-Blocker'}",
            fontsize=9, fontweight="bold"
        )
        ax.set_xlabel("Atom Col",   fontsize=6)
        ax.set_ylabel("Atom Row",   fontsize=6)
        ax.set_zlabel("Importance", fontsize=6)
        ax.tick_params(labelsize=5)
        ax.view_init(elev=30, azim=45)

    # Row 4 -- 4D scatter
    ax_4d = fig.add_subplot(gs[3, :])
    all_pos, all_imp, all_cls, all_grp = [], [], [], []
    for i, cam in enumerate(cam_maps):
        for node_idx, imp in enumerate(cam):
            all_pos.append(node_idx)
            all_imp.append(imp)
            all_cls.append(graph_samples[i].y.item())
            grp = min(node_idx * len(feat_group_names) // max(len(cam), 1),
                      len(feat_group_names) - 1)
            all_grp.append(grp)

    all_pos = np.array(all_pos)
    all_imp = np.array(all_imp)
    all_cls = np.array(all_cls)
    all_grp = np.array(all_grp)

    for gid, (gname, gcolor) in enumerate(
            zip(feat_group_names, feat_group_colors)):
        mask  = all_grp == gid
        sizes = np.where(all_cls[mask] == 1, 30, 10)
        ax_4d.scatter(all_pos[mask], all_imp[mask],
                      s=sizes, c=gcolor, alpha=0.5,
                      label=f"{gname} (large=Blocker)")

    ax_4d.set_title(
        "4D GNN GradCAM -- Atom Index (x) x Node Importance (y) x "
        "Class (marker size) x Feature Group (grayscale)",
        fontsize=10, fontweight="bold"
    )
    ax_4d.set_xlabel("Atom Index", fontsize=9)
    ax_4d.set_ylabel("GradCAM Importance (normalised)", fontsize=9)
    ax_4d.legend(fontsize=6, loc="upper right", ncol=2)
    ax_4d.grid(True, alpha=0.3)

    plt.savefig(f"{version_name}_gradcam.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {version_name}_gradcam.png")

# =============================================================
# VISUALISATION BLOCK 13 -- SCAFFOLD ANALYSIS
# =============================================================
print("\n" + "=" * 60)
print("  SCAFFOLD ANALYSIS")
print("=" * 60)

test_smiles_list = list(test_smiles_all[:len(test_graphs)])
scaffolds        = []
for smi in test_smiles_list:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        try:
            sc = MurckoScaffold.MurckoScaffoldSmiles(
                mol=mol, includeChirality=False)
            scaffolds.append(sc)
        except Exception:
            scaffolds.append("Unknown")
    else:
        scaffolds.append("Unknown")

scaffold_arr     = np.array(scaffolds)
unique_scaffolds = np.unique(scaffold_arr)
print(f"  Unique scaffolds in test set : {len(unique_scaffolds)}")

best_name    = max(gnn_results, key=lambda n: gnn_results[n]["ROC-AUC"])
res          = gnn_results[best_name]
scaffold_acc = {}

for sc in unique_scaffolds[:20]:
    mask = scaffold_arr == sc
    if mask.sum() >= 3:
        acc = accuracy_score(res["labels"][mask], res["preds"][mask])
        scaffold_acc[sc[:30]] = (acc, mask.sum())

if scaffold_acc:
    fig, ax = plt.subplots(figsize=(14, 6))
    sc_names = list(scaffold_acc.keys())
    sc_accs  = [scaffold_acc[s][0] for s in sc_names]
    sc_sizes = [scaffold_acc[s][1] for s in sc_names]
    bars     = ax.barh(sc_names, sc_accs, color="#009688",
                       edgecolor="black", alpha=0.8)
    for bar, n in zip(bars, sc_sizes):
        ax.text(bar.get_width() + 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"n={n}", va="center", fontsize=8)
    ax.axvline(0.87, color="red", linestyle="--",
               linewidth=1.5, label="Target (0.87)")
    ax.set_xlabel("Accuracy")
    ax.set_title(f"Scaffold-wise Accuracy\n{best_name} (Test Set)",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig("GNN_scaffold_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()

# =============================================================
# VISUALISATION BLOCK 14 -- PROBABILITY DISTRIBUTION
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, gnn_results.items()):
    blockers     = res["probs"][res["labels"] == 1]
    non_blockers = res["probs"][res["labels"] == 0]
    ax.hist(non_blockers, bins=40, alpha=0.6,
            color="#FF5722", label="Non-Blocker", edgecolor="black")
    ax.hist(blockers, bins=40, alpha=0.6,
            color="#9C27B0", label="Blocker",     edgecolor="black")
    ax.axvline(0.5, color="black", linestyle="--",
               linewidth=1.5, label="Decision boundary (0.5)")
    ax.set_title(f"Probability Distribution\n{name}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted Probability (Blocker)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle("GNN -- Prediction Confidence Distribution",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("GNN_probability_dist.png", dpi=150, bbox_inches="tight")
plt.show()

# =============================================================
# VISUALISATION BLOCK 15 -- SUMMARY TABLE
# =============================================================
fig, ax = plt.subplots(figsize=(16, 4))
ax.axis("off")
metrics_order = ["ROC-AUC", "PR-AUC", "Accuracy", "F1",
                 "Precision", "Recall", "Brier", "RMSE", "MAE"]
table_data    = []
for name in versions_gnn:
    row = [name.replace("GNN_", "")] + \
          [f"{gnn_results[name][m]:.4f}" for m in metrics_order]
    table_data.append(row)
col_labels = ["Version"] + metrics_order
table      = ax.table(
    cellText  = table_data,
    colLabels = col_labels,
    loc       = "center",
    cellLoc   = "center"
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 2.0)
for j in range(len(col_labels)):
    table[0, j].set_facecolor("#1a237e")
    table[0, j].set_text_props(color="white", fontweight="bold")
row_colors = ["#F3E5F5", "#FFF3E0", "#E0F2F1"]
for i, color in enumerate(row_colors):
    for j in range(len(col_labels)):
        table[i+1, j].set_facecolor(color)
plt.title("GNN Module -- Complete Results Summary",
          fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("GNN_summary_table.png", dpi=150, bbox_inches="tight")
plt.show()

# ── FINAL PRINT ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("  GNN MODULE -- FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"{'Version':<22} {'ROC-AUC':>8} {'PR-AUC':>8} "
      f"{'Acc':>7} {'F1':>7} {'Brier':>8} {'RMSE':>7} {'MAE':>7}")
print("-" * 78)
for name, res in gnn_results.items():
    target = "  <-- TARGET MET" if res["ROC-AUC"] >= 0.87 else ""
    print(f"{name:<22} {res['ROC-AUC']:>8.4f} {res['PR-AUC']:>8.4f} "
          f"{res['Accuracy']:>7.4f} {res['F1']:>7.4f} "
          f"{res['Brier']:>8.4f} {res['RMSE']:>7.4f} "
          f"{res['MAE']:>7.4f}{target}")

print("\n" + "=" * 60)
print("  STEP 12 COMPLETE -- ALL OUTPUTS SAVED")
print("=" * 60)
for f in [
    "GNN_loss_accuracy.png",      "GNN_auc_f1.png",
    "GNN_precision_recall_curves.png", "GNN_calibration_curves.png",
    "GNN_test_curves.png",         "GNN_all_metrics.png",
    "GNN_confusion_matrices.png",  "GNN_calibration_reliability.png",
    "GNN_threshold_analysis.png",  "GNN_bootstrap_ci.png",
    "GNN_tsne.png",                "GNN_probability_dist.png",
    "GNN_error_analysis.png",      "GNN_applicability_domain.png",
    "GNN_y_randomisation.png",     "GNN_Attention_gradcam.png",
    "GNN_NoAttention_gradcam.png", "GNN_Fingerprint_gradcam.png",
    "GNN_scaffold_analysis.png",   "GNN_summary_table.png",
]:
    print(f"  {f}")

  GNN MODULE -- COMPLETE WITH ALL FEATURES
Device                  : cpu
Node feature dim        : 55
Edge feature dim        : 13

  TRAINING : GNN_Attention

  Config: lr=0.001  wd=0.0001  epochs=50
  Epoch 010 | Train Loss: 0.4891  AUC: 0.8450  Acc: 0.7807 | Val Loss: 0.7859  AUC: 0.7663  Acc: 0.8105
  Epoch 020 | Train Loss: 0.4239  AUC: 0.8873  Acc: 0.8112 | Val Loss: 0.6378  AUC: 0.8253  Acc: 0.8140
  Epoch 030 | Train Loss: 0.2862  AUC: 0.9470  Acc: 0.8715 | Val Loss: 0.7723  AUC: 0.8256  Acc: 0.7767
  Epoch 040 | Train Loss: 0.2311  AUC: 0.9635  Acc: 0.8979 | Val Loss: 0.9104  AUC: 0.8223  Acc: 0.7919
  Epoch 050 | Train Loss: 0.1470  AUC: 0.9823  Acc: 0.9385 | Val Loss: 1.2180  AUC: 0.8376  Acc: 0.8023
  Val ROC-AUC : 0.8554

  Config: lr=0.0005  wd=0.0001  epochs=50
  Epoch 010 | Train Loss: 0.4492  AUC: 0.8709  Acc: 0.7931 | Val Loss: 0.7442  AUC: 0.7593  Acc: 0.7895
  Epoch 020 | Train Loss: 0.3453  AUC: 0.9240  Acc: 0.8425 | Val Loss: 0.6235  AUC: 0.8201  Acc: 0.7488
  Epo